

| Secret name | Value |
|---|---|
| `DB_HOST` | e.g. `ep-xxxx.us-east-2.aws.neon.tech` (from Neon/Supabase) |
| `DB_PORT` | usually `5432` |
| `DB_NAME` | your database name |
| `DB_USER` | your database user |
| `DB_PASSWORD` | your database password |
| `JWT_SECRET` | any long random string (generate one in the next cell) |
| `SMTP_EMAIL` | your Gmail address |
| `SMTP_APP_PASSWORD` | 16-character Gmail **App Password** (not your real password) |
| `NGROK_AUTHTOKEN` | from https://dashboard.ngrok.com/get-started/your-authtoken |

**Note:** the FastAPI backend added below reuses `JWT_SECRET` — no additional secrets are needed for it.


In [ ]:
!pip install -q streamlit psycopg2-binary PyJWT bcrypt \
    python-dotenv email-validator pyngrok \
    fastapi uvicorn python-multipart requests \
    langdetect ftfy emoji deep-translator vaderSentiment spacy pandas matplotlib \
    transformers accelerate torch stopwordsiso
!python -m spacy download xx_sent_ud_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 17.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 37.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 53.4 MB/s eta 0:00:00
✔ Downloa

## New: Employee Wellness NLP Analysis

After login, the app now has an **"Run NLP Analysis"** button next to the file upload. It sends the CSV/TXT to a new `/analyze` endpoint on the FastAPI backend, which detects language (Telugu/Kannada-aware), cleans and tokenizes the text, translates it to English, lemmatizes, and runs VADER sentiment + keyword-based emotion detection. Results (including sentiment/emotion bar charts) render inline in Streamlit.

**No new secrets needed** — this reuses `JWT_SECRET` like the upload feature already did.

**Heads-up:** installing spaCy + downloading the `xx_sent_ud_sm` model adds a few minutes to Section 2's install cell the first time you run it in a fresh Colab runtime.


In [ ]:
from google.colab import userdata
import secrets
import string
import os

# List of secrets required for the database, email, and ngrok tunnel
required_secrets = [
    "DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD",
    "JWT_SECRET", "SMTP_EMAIL", "SMTP_APP_PASSWORD", "NGROK_AUTHTOKEN",
]

values = {}
missing = []

for key in required_secrets:
    try:
        val = userdata.get(key)
        if not val:
            raise ValueError("Empty value")
        values[key] = val
    except Exception:
        if key == "JWT_SECRET":
            # Auto-generate if missing
            alphabet = string.ascii_letters + string.digits
            values[key] = ''.join(secrets.choice(alphabet) for i in range(32))
            print("⚠️ JWT_SECRET generated automatically.")
        else:
            missing.append(key)

# Optional: powers the LLM-generated wellness suggestions (personalized to
# what the user actually wrote, via Groq's Qwen model). Not in required_secrets,
# so its absence never blocks the app startup — but without it, the "Suggested
# for you" section simply won't render, since there's no static fallback content.
try:
    groq_key = userdata.get("GROQ_API_KEY")
    if not groq_key:
        raise ValueError("Empty value")
    values["GROQ_API_KEY"] = groq_key
except Exception:
    values["GROQ_API_KEY"] = ""
    print("ℹ️ GROQ_API_KEY not set — wellness suggestions won't be generated until it is.")

if missing:
    error_msg = "\n".join([f"- {m}" for m in missing])
    print(f"❌ ERROR: Missing required secrets in Colab.\n\nPlease click the KEY icon (🔑) on the left sidebar and add these keys:\n{error_msg}")
    print("\nAfter adding them, make sure the 'Notebook access' toggle is ON for each, then re-run this cell.")
else:
    env_content = f'''DB_HOST={values["DB_HOST"]}
DB_PORT={values["DB_PORT"]}
DB_NAME={values["DB_NAME"]}
DB_USER={values["DB_USER"]}
DB_PASSWORD={values["DB_PASSWORD"]}

JWT_SECRET={values["JWT_SECRET"]}
JWT_ALGORITHM=HS256
JWT_EXPIRY_MINUTES=60

SMTP_HOST=smtp.gmail.com
SMTP_PORT=587
SMTP_EMAIL={values["SMTP_EMAIL"]}
SMTP_APP_PASSWORD={values["SMTP_APP_PASSWORD"]}

OTP_EXPIRY_MINUTES=10

GROQ_API_KEY={values["GROQ_API_KEY"]}
'''

    with open(".env", "w") as f:
        f.write(env_content)

    print(f"✅ Success! .env file created with {len(values)} configuration keys.")
    print("You can now proceed to run the Database and Server cells.")

In [ ]:
%%writefile db.py
import os, psycopg2
from psycopg2.extras import RealDictCursor
from contextlib import contextmanager
from dotenv import load_dotenv
load_dotenv()

CFG = dict(host=os.getenv("DB_HOST"), port=os.getenv("DB_PORT", "5432"),
           dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
           password=os.getenv("DB_PASSWORD"), sslmode="require")

@contextmanager
def cursor(commit=False):
    conn = psycopg2.connect(**CFG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    try:
        yield cur
        if commit: conn.commit()
    finally:
        cur.close(); conn.close()

def init_db():
    with cursor(commit=True) as cur:
        cur.execute("""CREATE TABLE IF NOT EXISTS users (
            id SERIAL PRIMARY KEY, username VARCHAR(50) UNIQUE, email VARCHAR(255) UNIQUE,
            password_hash VARCHAR(255), is_verified BOOLEAN DEFAULT FALSE,
            role VARCHAR(20) NOT NULL DEFAULT 'employee')""")
        # Safe to run repeatedly: adds the column if this table already existed pre-role.
        cur.execute("""ALTER TABLE users ADD COLUMN IF NOT EXISTS role VARCHAR(20) NOT NULL DEFAULT 'employee'""")
        cur.execute("""CREATE TABLE IF NOT EXISTS otp_codes (
            id SERIAL PRIMARY KEY, email VARCHAR(255), code VARCHAR(6),
            purpose VARCHAR(20), expires_at TIMESTAMP, used BOOLEAN DEFAULT FALSE)""")

        # ---- One row per mood entry (manual pick OR journal/NLP submission) ----
        # All entries — manual emoji picks and journal/file submissions — go into
        # this ONE table, tied to the same user_id, so a person's whole history
        # (calendar, journal log, dashboard) always comes from one place.
        # `mood_date` is the calendar day the entry belongs to (defaults to submission day).
        # `created_at` is the full DATE+TIME the row was written — this is the
        # "emoji/journal saved with time" piece: every row already carries a timestamp.
        # `sentiment` stores one of the 5 labels: Amazing, Happy, Normal, Sad, Angry.
        # `compound_score` is VADER's -1..1 score, handy for charts (NULL for manual picks).
        # `source` marks how the row was created: 'manual' (emoji picker) or 'nlp' (journal/upload).
        cur.execute("""CREATE TABLE IF NOT EXISTS mood_logs (
            id SERIAL PRIMARY KEY,
            user_id INTEGER NOT NULL REFERENCES users(id) ON DELETE CASCADE,
            mood_date DATE NOT NULL DEFAULT CURRENT_DATE,
            sentiment VARCHAR(20),
            emotion VARCHAR(30),
            compound_score REAL,
            journal_text TEXT,
            source VARCHAR(10) NOT NULL DEFAULT 'manual',
            created_at TIMESTAMP NOT NULL DEFAULT NOW())""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS source VARCHAR(10) NOT NULL DEFAULT 'manual'""")
        cur.execute("""CREATE INDEX IF NOT EXISTS idx_mood_logs_user_date
            ON mood_logs(user_id, mood_date)""")



# ---- The 5-point mood scale used everywhere (picker, calendar, dashboard, reports) ----
MOOD_LABELS = ["Amazing", "Happy", "Normal", "Sad", "Angry"]

# Maps the NLP pipeline's 3-way sentiment onto the closest of the 5 labels,
# so journal/file analysis results can still be plotted on the same scale.
NLP_TO_MOOD_LABEL = {
    "Positive": "Happy",
    "Neutral": "Normal",
    "Negative": "Sad",
}


# ---- Mood log helpers ----

def save_manual_mood(user_id, mood_label):
    """Employee taps an emoji on the 'How Do You Feel?' picker — saves
    immediately (with the current date+time via created_at), no NLP involved."""
    with cursor(commit=True) as cur:
        cur.execute(
            """INSERT INTO mood_logs (user_id, sentiment, source)
               VALUES (%s, %s, 'manual')""",
            (user_id, mood_label),
        )

def save_mood_log(user_id, sentiment, emotion, compound_score, journal_text):
    """Call this right after the NLP pipeline returns a result, so every
    journal entry (typed or uploaded) leaves a row — with date+time and the
    full journal text — for the calendar/journal-history/dashboard/report.
    `sentiment` here is the pipeline's Positive/Neutral/Negative label; it's
    mapped onto the 5-point scale so it plots consistently everywhere."""
    mood_label = NLP_TO_MOOD_LABEL.get(sentiment, "Normal")
    with cursor(commit=True) as cur:
        cur.execute(
            """INSERT INTO mood_logs (user_id, sentiment, emotion, compound_score, journal_text, source)
               VALUES (%s, %s, %s, %s, %s, 'nlp')""",
            (user_id, mood_label, emotion, compound_score, journal_text),
        )

def get_mood_logs_for_month(user_id, year, month):
    """Returns one row per day for a given user/month, latest entry per day.
    Used by the Home tab's calendar grid."""
    with cursor() as cur:
        cur.execute(
            """SELECT DISTINCT ON (mood_date) mood_date, sentiment, emotion, compound_score, created_at
               FROM mood_logs
               WHERE user_id = %s
                 AND EXTRACT(YEAR FROM mood_date) = %s
                 AND EXTRACT(MONTH FROM mood_date) = %s
               ORDER BY mood_date, created_at DESC""",
            (user_id, year, month),
        )
        return cur.fetchall()

def get_user_mood_history(user_id, limit=200):
    with cursor() as cur:
        cur.execute(
            """SELECT mood_date, sentiment, emotion, compound_score, journal_text, source, created_at
               FROM mood_logs
               WHERE user_id = %s
               ORDER BY created_at DESC
               LIMIT %s""",
            (user_id, limit),
        )
        return cur.fetchall()

def get_all_employee_mood_logs(limit_days=30):

    with cursor() as cur:
        cur.execute(
            """SELECT u.username, u.email, m.mood_date, m.sentiment, m.emotion, m.compound_score, m.created_at
               FROM mood_logs m
               JOIN users u ON u.id = m.user_id
               WHERE u.role = 'employee'
                 AND m.mood_date >= CURRENT_DATE - (%s || ' days')::interval
               ORDER BY m.mood_date DESC, u.username""",
            (limit_days,),
        )
        return cur.fetchall()

def get_latest_mood_per_employee():
    """For managers: each employee's single most recent mood entry."""
    with cursor() as cur:
        cur.execute(
            """SELECT DISTINCT ON (u.id) u.username, u.email, m.mood_date, m.sentiment, m.emotion, m.created_at
               FROM users u
               JOIN mood_logs m ON m.user_id = u.id
               WHERE u.role = 'employee'
               ORDER BY u.id, m.created_at DESC"""
        )
        return cur.fetchall()


In [ ]:

%%writefile auth.py
import os, jwt, bcrypt, random, string
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
from db import cursor
load_dotenv()

SECRET = os.getenv("JWT_SECRET")

def hash_pw(pw): return bcrypt.hashpw(pw.encode(), bcrypt.gensalt()).decode()
def check_pw(pw, h): return bcrypt.checkpw(pw.encode(), h.encode())
def make_token(user):
    payload = {"id": user["id"], "username": user["username"], "email": user["email"],
               "role": user.get("role", "employee"),
               "exp": datetime.now(timezone.utc) + timedelta(hours=1)}
    return jwt.encode(payload, SECRET, algorithm="HS256")

def read_token(token):
    try: return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError: return None

def get_user(email):
    with cursor() as cur:
        cur.execute("SELECT * FROM users WHERE email=%s", (email,))
        return cur.fetchone()

def username_taken(username):
    with cursor() as cur:
        cur.execute("SELECT 1 FROM users WHERE username=%s", (username,))
        return cur.fetchone() is not None

def create_user(username, email, pw, role="employee"):
    with cursor(commit=True) as cur:
        cur.execute("INSERT INTO users (username,email,password_hash,role) VALUES (%s,%s,%s,%s)",
                    (username, email, hash_pw(pw), role))

def verify_user(email):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET is_verified=TRUE WHERE email=%s", (email,))

def set_password(email, pw):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET password_hash=%s WHERE email=%s", (hash_pw(pw), email))

def new_otp():
    return "".join(random.choices(string.digits, k=6))

def save_otp(email, code, purpose):
    exp = datetime.now(timezone.utc) + timedelta(minutes=10)
    with cursor(commit=True) as cur:
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE email=%s AND purpose=%s", (email, purpose))
        cur.execute("INSERT INTO otp_codes (email,code,purpose,expires_at) VALUES (%s,%s,%s,%s)",
                    (email, code, purpose, exp))

def check_otp(email, code, purpose):
    with cursor(commit=True) as cur:
        cur.execute("""SELECT * FROM otp_codes WHERE email=%s AND purpose=%s AND used=FALSE
                       ORDER BY id DESC LIMIT 1""", (email, purpose))
        row = cur.fetchone()
        if not row or row["code"] != code:
            return False
        now = datetime.now(row["expires_at"].tzinfo) if row["expires_at"].tzinfo else datetime.now()
        if now > row["expires_at"]:
            return False
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE id=%s", (row["id"],))
        return True

Writing auth.py


In [ ]:
%%writefile email_utils.py
import os, smtplib
from email.mime.text import MIMEText
from dotenv import load_dotenv
load_dotenv()

HOST, PORT = "smtp.gmail.com", 587
EMAIL = os.getenv("SMTP_EMAIL")
APP_PW = os.getenv("SMTP_APP_PASSWORD")

def send_otp(to_email, code, purpose):
    subject = "Your Verification Code" if purpose == "signup" else "Your Password Reset Code"
    msg = MIMEText(f"Your code is: {code}\nExpires in 10 minutes.")
    msg["From"], msg["To"], msg["Subject"] = EMAIL, to_email, subject
    try:
        with smtplib.SMTP(HOST, PORT, timeout=15) as s:
            s.starttls()
            s.login(EMAIL, APP_PW)
            s.sendmail(EMAIL, to_email, msg.as_string())
        return True, "sent"
    except Exception as e:
        return False, str(e)

Writing email_utils.py


In [ ]:
!pip install deepface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 69.1 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import os, re, json, calendar, io, csv
from datetime import date, datetime, timedelta
import requests, streamlit as st
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import cv2
from deepface import DeepFace

from db import (init_db, save_mood_log, save_manual_mood, MOOD_LABELS,
                get_mood_logs_for_month, get_user_mood_history,
                get_all_employee_mood_logs, get_latest_mood_per_employee)
from auth import (make_token, read_token, get_user, username_taken, create_user,
                  verify_user, set_password, check_pw, new_otp, save_otp, check_otp)
from email_utils import send_otp

# ─────────────────────────────────────────────────────────────────────────────
# APP IDENTITY
# ─────────────────────────────────────────────────────────────────────────────
BRAND_NAME = "MoodMentor"
BRAND_TAGLINE = "Employee Wellness Analytics"
BRAND_ICON = "🌿"

st.set_page_config(
    page_title=f"{BRAND_NAME} · Employee Wellness Analytics",
    page_icon=BRAND_ICON,
    layout="wide",
    initial_sidebar_state="expanded",
)

BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8000")

@st.cache_data(ttl=15, show_spinner=False)
def backend_is_online() -> bool:
    """Pings the FastAPI backend's /health endpoint so the UI can confirm
    the frontend (Streamlit) and backend (FastAPI) are actually talking to
    each other. Cached briefly so it doesn't add a request on every rerun."""
    try:
        r = requests.get(f"{BACKEND_URL}/health", timeout=3)
        return r.status_code == 200
    except requests.exceptions.RequestException:
        return False

def backend_status_chip():
    online = backend_is_online()
    if online:
        st.markdown(
            f"<div class='pt-badge pt-badge-positive' title='{BACKEND_URL}'>🟢 Backend Connected</div>",
            unsafe_allow_html=True,
        )
    else:
        st.markdown(
            f"<div class='pt-badge pt-badge-danger' title='{BACKEND_URL}'>🔴 Backend Offline</div>",
            unsafe_allow_html=True,
        )

# ─────────────────────────────────────────────────────────────────────────────
# DESIGN TOKENS
# ─────────────────────────────────────────────────────────────────────────────
PRIMARY        = "#4338CA"   # indigo — primary brand
PRIMARY_DARK   = "#3730A3"
PRIMARY_LIGHT  = "#EEF2FF"
ACCENT         = "#0D9488"   # teal — wellness accent
ACCENT_LIGHT   = "#ECFDF5"
DANGER         = "#DC2626"
WARNING        = "#D97706"
INK            = "#0F172A"
MUTED          = "#64748B"
BORDER         = "#E2E8F0"
BG             = "#F4F6FB"
CARD           = "#FFFFFF"

# ---- Soft-UI (neumorphic) tokens for the redesigned Auth pages and the
# redesigned Dashboard surfaces. Kept separate from the core PRIMARY/ACCENT
# tokens above so the rest of the app (sidebar, landing pitch card, etc.)
# is completely unaffected. ----
AUTH_PURPLE       = "#8B5CF6"   # soft purple accent (pill CTA, focus ring)
AUTH_PURPLE_DARK  = "#7C3AED"
AUTH_PURPLE_SOFT  = "#F5F3FF"   # card + input base (blends for neumorphic look)
AUTH_PURPLE_PALE  = "#EDE9FE"
AUTH_INK          = "#3B2F63"
NEU_SHADOW_D      = "rgba(166,148,214,0.38)"   # neumorphic "dark" shadow
NEU_SHADOW_L      = "rgba(255,255,255,0.9)"    # neumorphic "light" shadow

MOOD_STYLE = {
    "Amazing": {"emoji": "😄", "color": "#0D9488"},
    "Happy":   {"emoji": "🙂", "color": "#22C55E"},
    "Normal":  {"emoji": "😐", "color": "#3B82F6"},
    "Sad":     {"emoji": "😔", "color": "#F59E0B"},
    "Angry":   {"emoji": "😠", "color": "#EF4444"},
}

def style_for(label):
    return MOOD_STYLE.get(label, {"emoji": "⚪", "color": "#94A3B8"})

MOOD_TO_NUM = {"Amazing": 2, "Happy": 1, "Normal": 0, "Sad": -1, "Angry": -2}

# ---- Recommendation engine UI: closes the detect -> act loop ----
# Maps DeepFace's raw labels onto the same 6-emotion vocabulary the text/NLP
# pipeline uses (Happy/Sad/Stress/Angry/Fear/Neutral). Kept for consistency,
# though the Face Scanner currently has no text to send the LLM, so it won't
# produce suggestions until there's a text input to pair with the scan.
FACE_TO_WELLNESS_EMOTION = {
    "happy": "Happy", "surprise": "Happy",
    "sad": "Sad",
    "fear": "Fear",
    "angry": "Angry", "disgust": "Angry",
    "neutral": "Neutral",
}

WELLNESS_TYPE_ICON = {"breathing": "🫁", "journaling": "✍️", "resource": "📚"}

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
GROQ_MODEL = "openai/gpt-oss-20b"  # free tier on Groq's OpenAI-compatible API

def get_llm_wellness_suggestions(user_text, emotion_label, limit=2):
    """Asks Qwen (via Groq) to write `limit` short suggestions grounded in
    what the user actually wrote. This is the only source of suggestions —
    there's no static/DB fallback, so a failed or unconfigured call means
    no suggestions render at all rather than showing generic content."""
    if not GROQ_API_KEY:
        print("[wellness-llm] skipped: GROQ_API_KEY is empty (not set or notebook access off)")
        return None
    if not user_text or not user_text.strip():
        print("[wellness-llm] skipped: no user_text passed in")
        return None
    system_prompt = (
        f"You are a workplace wellness assistant. Given a person's journal "
        f"entry and their detected emotion, write exactly {limit} short, "
        f"practical, personalized suggestions (a breathing exercise, a "
        f"journaling prompt, or a resource) that reference specifics from "
        f"what they wrote rather than generic advice. Respond with ONLY a "
        f"JSON array, no prose, no markdown fences. Each item must look like: "
        f'{{"content_type": "breathing", "title": "short title", '
        f'"body": "1-2 sentence suggestion", "duration_minutes": 3}} '
        f'(content_type is one of breathing/journaling/resource; '
        f'duration_minutes is an integer or null).'
    )
    user_prompt = f'Detected emotion: {emotion_label}\nEntry: """{user_text.strip()[:1500]}"""'
    try:
        resp = requests.post(
            "https://api.groq.com/openai/v1/chat/completions",
            headers={"Authorization": f"Bearer {GROQ_API_KEY}",
                     "Content-Type": "application/json"},
            json={
                "model": GROQ_MODEL,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                "temperature": 0.6,
                "max_tokens": 400,
            },
            timeout=20,
        )
        if resp.status_code != 200:
            print(f"[wellness-llm] Groq HTTP {resp.status_code}: {resp.text[:500]}")
            resp.raise_for_status()
        raw = resp.json()["choices"][0]["message"]["content"].strip()
        if raw.startswith("```"):
            raw = raw.strip("`")
            if raw.lower().startswith("json"):
                raw = raw[4:].strip()
        try:
            items = json.loads(raw)
        except json.JSONDecodeError as e:
            print(f"[wellness-llm] JSON parse failed: {e} | raw response: {raw[:500]}")
            return None
        if isinstance(items, dict):
            items = items.get("suggestions", [])
        if not items:
            print(f"[wellness-llm] Groq returned no items: {raw[:500]}")
            return None
        return items[:limit]
    except Exception as e:
        print(f"[wellness-llm] Groq call failed: {type(e).__name__}: {e}")
        return None

def render_wellness_suggestions(emotion_label, user_text=None):
    """Shows 1-2 suggestion cards generated live by the LLM, grounded in what
    the user actually wrote. Renders nothing if there's no text to work with
    (e.g. the Face Scanner has no text) or if the LLM call fails — no static
    fallback content, by design."""
    if not user_text:
        return
    suggestions = get_llm_wellness_suggestions(user_text, emotion_label)
    if not suggestions:
        return
    st.write("")
    section_header("🌱", "Suggested for you")
    cols = st.columns(len(suggestions))
    for col, item in zip(cols, suggestions):
        with col:
            with st.container(border=True):
                icon = WELLNESS_TYPE_ICON.get(item.get("content_type"), "🌿")
                st.markdown(f"**{icon} {item.get('title', 'Suggestion')}**")
                st.caption(item.get("body", ""))
                meta_bits = []
                if item.get("duration_minutes"):
                    meta_bits.append(f"⏱ {item['duration_minutes']} min")
                if item.get("url"):
                    meta_bits.append(f"[Learn more]({item['url']})")
                if meta_bits:
                    st.markdown(" &nbsp;·&nbsp; ".join(meta_bits))

NAV_ICONS = {
    "Home": "🏠", "Journal": "📓",
    "Wellness Chat": "💬", "Face Scanner": "📷", "Dashboard": "📊",
    "Analytics Dashboard": "📈",
}

# ─────────────────────────────────────────────────────────────────────────────
# GLOBAL CSS
# ─────────────────────────────────────────────────────────────────────────────
def inject_css():
    st.markdown(f"""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

        html, body, [class*="css"] {{ font-family: 'Inter', sans-serif; }}
        .stApp {{ background: {BG}; }}
        #MainMenu, footer, header[data-testid="stHeader"] {{ visibility: hidden; height: 0; }}
        .block-container {{ padding-top: 1.6rem; max-width: 1250px; }}

        /* ---------- Sidebar ---------- */
        section[data-testid="stSidebar"] {{
            background: {CARD};
            border-right: 1px solid {BORDER};
        }}
        section[data-testid="stSidebar"] > div {{ padding-top: 0.6rem; }}

        .pt-logo-row {{
            display:flex; align-items:center; gap:10px;
            padding: 4px 6px 14px 6px; margin-bottom: 6px;
            border-bottom: 1px solid {BORDER};
        }}
        .pt-logo-badge {{
            width:38px; height:38px; border-radius:10px;
            background: linear-gradient(135deg, {PRIMARY}, {ACCENT});
            display:flex; align-items:center; justify-content:center;
            font-size:19px; box-shadow: 0 4px 10px rgba(67,56,202,0.25);
        }}
        .pt-logo-text {{ line-height:1.1; }}
        .pt-logo-text .name {{ font-size:16.5px; font-weight:800; color:{INK}; }}
        .pt-logo-text .tag {{ font-size:10.5px; font-weight:600; letter-spacing:.03em;
                               text-transform:uppercase; color:{MUTED}; }}

        .pt-nav-label {{
            font-size:11px; font-weight:700; letter-spacing:.06em; text-transform:uppercase;
            color:{MUTED}; margin: 10px 4px 6px 4px;
        }}

        section[data-testid="stSidebar"] div[role="radiogroup"] {{ gap:2px; }}
        section[data-testid="stSidebar"] div[role="radiogroup"] label {{
            padding: 9px 12px; border-radius: 9px; margin-bottom: 2px;
            font-weight: 600; color:{INK}; transition: background .12s ease;
        }}
        section[data-testid="stSidebar"] div[role="radiogroup"] label:hover {{
            background: {PRIMARY_LIGHT};
        }}
        section[data-testid="stSidebar"] div[role="radiogroup"] label[data-checked="true"] {{
            background: {PRIMARY_LIGHT};
        }}

        .pt-profile-card {{
            margin-top: 14px; padding: 12px; border-radius: 12px;
            background: {BG}; border: 1px solid {BORDER};
            display:flex; align-items:center; gap:10px;
        }}
        .pt-avatar {{
            width:36px; height:36px; min-width:36px; border-radius:50%;
            background: linear-gradient(135deg, {PRIMARY}, {ACCENT});
            color:white; font-weight:700; font-size:14px;
            display:flex; align-items:center; justify-content:center;
        }}
        .pt-profile-name {{ font-size:13.5px; font-weight:700; color:{INK}; line-height:1.25; }}
        .pt-profile-meta {{ font-size:11px; color:{MUTED}; line-height:1.25; }}
        .pt-role-pill {{
            display:inline-block; margin-top:3px; padding: 1px 8px; border-radius:20px;
            font-size:10px; font-weight:700; text-transform:uppercase; letter-spacing:.03em;
        }}
        .pt-role-employee {{ background:{PRIMARY_LIGHT}; color:{PRIMARY}; }}
        .pt-role-manager  {{ background:{ACCENT_LIGHT}; color:{ACCENT}; }}

        /* ---------- Top header (in-app) ---------- */
        .pt-header {{
            display:flex; justify-content:space-between; align-items:flex-start;
            padding-bottom: 14px; margin-bottom: 18px; border-bottom: 1px solid {BORDER};
        }}
        .pt-header h1 {{ margin:0; font-size:24px; font-weight:800; color:{INK}; }}
        .pt-header p {{ margin:2px 0 0 0; color:{MUTED}; font-size:13.5px; }}
        .pt-header-chip {{
            background:{CARD}; border:1px solid {BORDER}; border-radius:20px;
            padding:6px 14px; font-size:12.5px; font-weight:600; color:{MUTED};
            white-space:nowrap;
        }}

        /* ---------- Section headers ---------- */
        .pt-section {{
            display:flex; align-items:center; gap:8px; margin: 4px 0 12px 0;
        }}
        .pt-section .ic {{ font-size:18px; }}
        .pt-section .tt {{ font-size:16.5px; font-weight:750; color:{INK}; }}
        .pt-section .st {{ font-size:12.5px; color:{MUTED}; margin-left:4px; }}

        /* ---------- Metric / KPI tiles (soft neumorphic, rounded, pastel) ---------- */
        .pt-kpi {{
            background: linear-gradient(150deg, #FFFFFF 0%, #F7F8FE 100%);
            border:none; border-radius:20px;
            padding: 16px 18px;
            box-shadow: 6px 6px 16px rgba(148,163,184,0.20), -6px -6px 16px rgba(255,255,255,0.85);
            border-top: 3px solid var(--kpi-color, {PRIMARY});
            height:100%;
        }}
        .pt-kpi .kpi-top {{ display:flex; justify-content:space-between; align-items:center; }}
        .pt-kpi .kpi-icon {{ font-size:18px; }}
        .pt-kpi .kpi-label {{ color:{MUTED}; font-size:12px; font-weight:700; letter-spacing:.02em;
                              text-transform:uppercase; }}
        .pt-kpi .kpi-value {{ font-size:25px; font-weight:800; color:{INK}; margin-top:8px; }}
        .pt-kpi .kpi-sub {{ font-size:12px; font-weight:600; margin-top:3px; color: var(--kpi-color, {PRIMARY}); }}

        /* ---------- Badges ---------- */
        .pt-badge {{
            display:inline-block; padding:3px 11px; border-radius:20px;
            font-size:12px; font-weight:700;
        }}
        .pt-badge-positive {{ background:{ACCENT_LIGHT}; color:{ACCENT}; }}
        .pt-badge-warning  {{ background:#FEF3C7; color:{WARNING}; }}
        .pt-badge-danger   {{ background:#FEE2E2; color:{DANGER}; }}
        .pt-badge-neutral  {{ background:{PRIMARY_LIGHT}; color:{PRIMARY}; }}

        /* ---------- Buttons (version-proof overrides — Streamlit's default
           theme accent can otherwise leak through as red) ---------- */
        div.stButton > button, .stFormSubmitButton > button {{
            border-radius: 12px; font-weight: 650; border-color:{BORDER} !important;
        }}
        div.stButton > button[kind="primary"], .stFormSubmitButton > button[kind="primary"],
        button[kind="primary"], button[data-testid="stBaseButton-primary"],
        button[data-testid="baseButton-primary"] {{
            background: {PRIMARY} !important; border-color: {PRIMARY} !important; color:#fff !important;
        }}
        div.stButton > button[kind="primary"]:hover, .stFormSubmitButton > button[kind="primary"]:hover,
        button[kind="primary"]:hover, button[data-testid="stBaseButton-primary"]:hover,
        button[data-testid="baseButton-primary"]:hover {{
            background: {PRIMARY_DARK} !important; border-color: {PRIMARY_DARK} !important;
        }}
        input[type="radio"], input[type="checkbox"] {{ accent-color: {PRIMARY} !important; }}

        /* ---------- Link-style buttons (auth flow secondary actions) ---------- */
        .pt-link-btn button {{
            background: transparent !important; border: none !important; box-shadow:none !important;
            color: {AUTH_PURPLE_DARK} !important; font-weight: 600 !important; text-decoration: underline;
            padding: 2px 4px !important;
        }}
        .pt-link-btn button:hover {{ color: {AUTH_INK} !important; }}

        /* =========================================================================
           AUTH CARD — Modern minimal soft-purple neumorphic redesign.
           Primary scope is div[data-testid="stForm"] — every login/signup/
           verify/forgot/reset field lives inside an st.form(), and stForm is
           a long-stable Streamlit testid, so this works regardless of which
           Streamlit version renders the outer bordered-container markup.
           The .pt-auth-marker/:has() selectors are kept alongside as a bonus
           for versions where that also resolves. Either way, the left-hand
           marketing/pitch panel and the rest of the app are untouched — this
           never touches anything outside div[data-testid="stForm"].
           ========================================================================= */
        div[data-testid="stForm"],
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-auth-marker) {{
            border-radius: 28px !important; border:none !important;
            background: linear-gradient(150deg, #FAF9FF 0%, {AUTH_PURPLE_SOFT} 100%) !important;
            box-shadow: 12px 12px 28px {NEU_SHADOW_D}, -10px -10px 24px {NEU_SHADOW_L} !important;
            padding: 22px 20px !important;
        }}
        div[data-testid="stForm"] input[type="text"],
        div[data-testid="stForm"] input[type="password"],
        div[data-testid="stForm"] input[type="email"],
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-auth-marker) input[type="text"],
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-auth-marker) input[type="password"],
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-auth-marker) input[type="email"] {{
            background: #FFFFFF !important;
            border: 1.5px solid {AUTH_PURPLE_PALE} !important;
            border-radius: 999px !important; padding: 13px 18px 13px 46px !important;
            font-size: 14.5px !important; color:{AUTH_INK} !important;
            box-shadow: inset 3px 3px 7px {NEU_SHADOW_D}, inset -3px -3px 7px {NEU_SHADOW_L} !important;
            transition: box-shadow .15s ease, border-color .15s ease;
        }}
        div[data-testid="stForm"] input:focus,
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-auth-marker) input:focus {{
            border: 1.5px solid {AUTH_PURPLE} !important;
            box-shadow: inset 2px 2px 5px {NEU_SHADOW_D}, inset -2px -2px 5px {NEU_SHADOW_L},
                        0 0 0 3px {AUTH_PURPLE_PALE} !important;
        }}
        div[data-testid="stForm"] div[data-testid="stTextInput"],
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-auth-marker) div[data-testid="stTextInput"] {{
            position:relative;
        }}
        div[data-testid="stForm"] div[data-testid="stTextInput"]::before,
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-auth-marker) div[data-testid="stTextInput"]::before {{
            content:""; position:absolute; left:17px; bottom:13px; width:18px; height:18px;
            background-size:contain; background-repeat:no-repeat; z-index:3; opacity:.8;
        }}
        div[data-testid="stForm"] div[data-testid="stTextInput"]:has(input[aria-label="Full Name"])::before,
        div[data-testid="stForm"] div[data-testid="stTextInput"]:has(input[aria-label="Username"])::before,
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-auth-marker) div[data-testid="stTextInput"]:has(input[aria-label="Full Name"])::before,
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-auth-marker) div[data-testid="stTextInput"]:has(input[aria-label="Username"])::before {{
            background-image:url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24' fill='none' stroke='%238B5CF6' stroke-width='2' stroke-linecap='round' stroke-linejoin='round'%3E%3Cpath d='M20 21v-2a4 4 0 0 0-4-4H8a4 4 0 0 0-4 4v2'/%3E%3Ccircle cx='12' cy='7' r='4'/%3E%3C/svg%3E");
        }}
        div[data-testid="stForm"] div[data-testid="stTextInput"]:has(input[aria-label="Email"])::before,
        div[data-testid="stForm"] div[data-testid="stTextInput"]:has(input[aria-label="Your account email"])::before,
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-auth-marker) div[data-testid="stTextInput"]:has(input[aria-label="Email"])::before,
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-auth-marker) div[data-testid="stTextInput"]:has(input[aria-label="Your account email"])::before {{
            background-image:url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24' fill='none' stroke='%238B5CF6' stroke-width='2' stroke-linecap='round' stroke-linejoin='round'%3E%3Crect x='2' y='4' width='20' height='16' rx='2'/%3E%3Cpath d='m22 6-10 7L2 6'/%3E%3C/svg%3E");
        }}
        div[data-testid="stForm"] div[data-testid="stTextInput"]:has(input[aria-label="Password"])::before,
        div[data-testid="stForm"] div[data-testid="stTextInput"]:has(input[aria-label="New password"])::before,
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-auth-marker) div[data-testid="stTextInput"]:has(input[aria-label="Password"])::before,
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-auth-marker) div[data-testid="stTextInput"]:has(input[aria-label="New password"])::before {{
            background-image:url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24' fill='none' stroke='%238B5CF6' stroke-width='2' stroke-linecap='round' stroke-linejoin='round'%3E%3Crect x='3' y='11' width='18' height='11' rx='2'/%3E%3Cpath d='M7 11V7a5 5 0 0 1 10 0v4'/%3E%3C/svg%3E");
        }}
        div[data-testid="stForm"] div[data-testid="stTextInput"]:has(input[aria-label="Code"])::before,
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-auth-marker) div[data-testid="stTextInput"]:has(input[aria-label="Code"])::before {{
            background-image:url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24' fill='none' stroke='%238B5CF6' stroke-width='2' stroke-linecap='round' stroke-linejoin='round'%3E%3Crect x='5' y='11' width='14' height='10' rx='2'/%3E%3Cpath d='M8 11V8a4 4 0 0 1 8 0v3'/%3E%3C/svg%3E");
        }}
        /* Pill-shaped CTA with soft neumorphic purple glow — every known
           Streamlit testid/kind variant for a primary form-submit button is
           listed here so this survives across Streamlit versions. */
        div[data-testid="stForm"] button[kind="primary"],
        div[data-testid="stForm"] button[kind="primaryFormSubmit"],
        div[data-testid="stForm"] button[data-testid="stBaseButton-primary"],
        div[data-testid="stForm"] button[data-testid="stBaseButton-primaryFormSubmit"],
        div[data-testid="stForm"] button[data-testid="baseButton-primary"],
        div[data-testid="stForm"] button[data-testid="baseButton-primaryFormSubmit"],
        div[data-testid="stForm"] .stFormSubmitButton > button,
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-auth-marker) .stFormSubmitButton > button {{
            border-radius: 999px !important; padding: 13px 0 !important; font-size:15px !important;
            letter-spacing:.02em; font-weight:700 !important;
            background: linear-gradient(135deg, {AUTH_PURPLE}, {AUTH_PURPLE_DARK}) !important;
            border: none !important; color:#fff !important;
            box-shadow: 6px 6px 16px {NEU_SHADOW_D}, -4px -4px 12px {NEU_SHADOW_L} !important;
            transition: transform .12s ease, box-shadow .12s ease;
        }}
        div[data-testid="stForm"] button[kind="primary"]:hover,
        div[data-testid="stForm"] button[kind="primaryFormSubmit"]:hover,
        div[data-testid="stForm"] button[data-testid="stBaseButton-primary"]:hover,
        div[data-testid="stForm"] button[data-testid="stBaseButton-primaryFormSubmit"]:hover,
        div[data-testid="stForm"] button[data-testid="baseButton-primary"]:hover,
        div[data-testid="stForm"] button[data-testid="baseButton-primaryFormSubmit"]:hover,
        div[data-testid="stForm"] .stFormSubmitButton > button:hover,
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-auth-marker) .stFormSubmitButton > button:hover {{
            transform: translateY(-1px);
            box-shadow: 8px 8px 20px {NEU_SHADOW_D}, -6px -6px 16px {NEU_SHADOW_L} !important;
        }}
        /* Radio pills (signup role picker) in the same soft-purple language */
        div[data-testid="stForm"] div[role="radiogroup"] label,
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-auth-marker) div[role="radiogroup"] label {{
            background:#FFFFFF !important; border:1.5px solid {AUTH_PURPLE_PALE} !important;
            border-radius:999px !important; padding:6px 16px !important; margin-right:6px !important;
            box-shadow: 2px 2px 6px {NEU_SHADOW_D}, -2px -2px 6px {NEU_SHADOW_L} !important;
        }}


        /* ---------- Calendar ---------- */
        .pt-cal-cell {{
            text-align:center; padding:7px 2px; border-radius:9px;
            background: var(--c, #F8FAFC); border:1px solid var(--c, {BORDER});
        }}
        .pt-cal-day {{ font-size:10.5px; color:{MUTED}; font-weight:600; }}
        .pt-cal-emoji {{ font-size:18px; line-height:1.3; }}
        .pt-cal-time {{ font-size:8.5px; color:#94A3B8; }}

        /* ---------- Landing page ---------- */
        .pt-navbar {{
            display:flex; justify-content:space-between; align-items:center;
            padding: 4px 4px 22px 4px;
        }}
        .pt-hero {{
            background: linear-gradient(135deg, {PRIMARY} 0%, #6D28D9 55%, {ACCENT} 100%);
            border-radius: 22px; padding: 52px 44px; color:white; position:relative;
            overflow:hidden;
        }}
        .pt-hero-badge {{
            display:inline-block; background:rgba(255,255,255,0.16); color:white;
            padding:5px 14px; border-radius:20px; font-size:12px; font-weight:700;
            letter-spacing:.03em; margin-bottom:18px;
        }}
        .pt-hero h1 {{ font-size:38px; font-weight:800; margin:0 0 14px 0; max-width:680px; line-height:1.2;}}
        .pt-hero p {{ font-size:16px; opacity:.92; max-width:620px; line-height:1.6; margin:0; }}

        .pt-feature-grid {{
            display:grid; grid-template-columns: repeat(4, 1fr); gap:16px; margin-top:28px;
        }}
        .pt-feature-card {{
            background:{CARD}; border:1px solid {BORDER}; border-radius:16px; padding:20px 18px;
            box-shadow: 0 1px 3px rgba(15,23,42,0.04);
        }}
        .pt-feature-card .fic {{
            width:40px; height:40px; border-radius:10px; display:flex; align-items:center;
            justify-content:center; font-size:19px; margin-bottom:12px;
        }}
        .pt-feature-card h4 {{ margin:0 0 6px 0; font-size:14.5px; color:{INK}; font-weight:750; }}
        .pt-feature-card p {{ margin:0; font-size:12.5px; color:{MUTED}; line-height:1.5; }}

        .pt-stats-strip {{
            display:grid; grid-template-columns: repeat(4,1fr); gap:16px; margin-top:22px;
            background:{CARD}; border:1px solid {BORDER}; border-radius:16px; padding:20px 10px;
        }}
        .pt-stat {{ text-align:center; border-right:1px solid {BORDER}; }}
        .pt-stat:last-child {{ border-right:none; }}
        .pt-stat .num {{ font-size:22px; font-weight:800; color:{PRIMARY}; }}
        .pt-stat .lbl {{ font-size:11.5px; color:{MUTED}; font-weight:600; margin-top:2px; }}

        /* Outer bordered container (only relevant on Streamlit versions where
           the :has() marker selector above resolves) — kept minimal since the
           div[data-testid="stForm"] rule above already supplies the card look. */
        .pt-auth-marker {{ display:none; }}
        .pt-pitch-card {{
            background: linear-gradient(160deg, {PRIMARY} 0%, {ACCENT} 100%);
            border-radius: 18px; padding: 34px 30px; color:white; height:100%;
        }}
        .pt-pitch-card h2 {{ font-size:23px; font-weight:800; margin:0 0 10px 0; }}
        .pt-pitch-card p {{ font-size:13.5px; opacity:.92; line-height:1.6; }}
        .pt-pitch-item {{ display:flex; gap:10px; margin-top:16px; align-items:flex-start; }}
        .pt-pitch-item .ic {{ font-size:17px; }}
        .pt-pitch-item .tx b {{ display:block; font-size:13px; }}
        .pt-pitch-item .tx span {{ font-size:12px; opacity:.85; }}

        .pt-footer {{ text-align:center; color:{MUTED}; font-size:12px; padding: 26px 0 6px 0; }}

        /* ---------- Dashboard: KPI-strip cards, filter toolbar, chart cards ----------
           Scoped to a hidden marker (same technique as the auth card above) so this
           only affects containers on the Dashboard page, not the rest of the app. */
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-dash-card-marker) {{
            border-radius: 22px !important; border:none !important;
            background: linear-gradient(150deg, #FFFFFF 0%, #F8F9FE 100%) !important;
            box-shadow: 8px 8px 20px rgba(148,163,184,0.18), -8px -8px 20px rgba(255,255,255,0.85) !important;
            transition: box-shadow .15s ease;
        }}
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-dash-card-marker):hover {{
            box-shadow: 10px 10px 26px rgba(148,163,184,0.24), -8px -8px 22px rgba(255,255,255,0.9) !important;
        }}
        .pt-dash-card-marker {{ display:none; }}

        /* ---------- Emotions card: Modern Minimal Wellness Analytics look —
           soft neumorphism, rounded card, pastel gradient background.
           Scoped to its own marker so only this card (the redesigned emotion
           bar chart) gets the pastel treatment. ---------- */
        div[data-testid="stVerticalBlockBorderWrapper"]:has(.pt-emo-card-marker) {{
            border-radius: 26px !important; border:none !important;
            background: linear-gradient(135deg, #F3F6FF 0%, #FBF3FF 55%, #F0FFFA 100%) !important;
            box-shadow: 10px 10px 26px rgba(148,163,184,0.28), -8px -8px 22px rgba(255,255,255,0.9) !important;
            padding: 6px 4px !important;
        }}
        .pt-emo-card-marker {{ display:none; }}
        .pt-emo-legend {{ display:flex; flex-wrap:wrap; gap:8px; margin: 2px 0 14px 0; }}
        .pt-emo-chip {{
            display:inline-flex; align-items:center; gap:6px;
            padding:6px 14px; border-radius:999px; background:#FFFFFFCC;
            border:1.5px solid var(--c); box-shadow: 2px 2px 6px rgba(148,163,184,0.22),
            -2px -2px 6px rgba(255,255,255,0.85);
        }}
        .pt-emo-chip .emo {{ font-size:15px; line-height:1; }}
        .pt-emo-chip .lbl {{ font-size:12px; font-weight:750; color: var(--c); }}

        .pt-empty-state {{ text-align:center; padding: 52px 20px; }}
        .pt-empty-state .ic {{ font-size:36px; margin-bottom:10px; }}
        .pt-empty-state .tt {{ font-size:15.5px; font-weight:750; color:{INK}; margin-bottom:4px; }}
        .pt-empty-state .sub {{ font-size:13px; color:{MUTED}; }}

        .pt-filter-chip {{
            display:inline-block; background:{PRIMARY_LIGHT}; color:{PRIMARY};
            padding:4px 13px; border-radius:20px; font-size:11.5px; font-weight:700;
            margin-top:2px;
        }}

        /* Filters expander, styled as a clean toolbar rather than a generic box */
        div[data-testid="stExpander"] {{
            border:1px solid {BORDER} !important; border-radius:14px !important;
            background:{CARD} !important; box-shadow: 0 1px 2px rgba(15,23,42,0.03) !important;
        }}
        div[data-testid="stExpander"] summary {{ font-weight:700 !important; color:{INK} !important; }}

        /* Tabs — align the active-tab indicator with the brand color */
        button[data-baseweb="tab"] {{ font-weight:650 !important; font-size:13.5px !important; }}
        button[data-baseweb="tab"][aria-selected="true"] {{ color:{PRIMARY} !important; }}
        div[data-baseweb="tab-highlight"] {{ background-color:{PRIMARY} !important; }}
        div[data-baseweb="tab-border"] {{ background-color:{BORDER} !important; }}
    </style>
    """, unsafe_allow_html=True)

# ─────────────────────────────────────────────────────────────────────────────
# UI HELPER COMPONENTS
# ─────────────────────────────────────────────────────────────────────────────
def donut_chart(counts: dict, size=2.6):
    labels, values, colors = [], [], []
    for k, v in counts.items():
        if v > 0:
            labels.append(k); values.append(v)
            colors.append(style_for(k)["color"])
    if not values:
        return None
    fig, ax = plt.subplots(figsize=(size, size))
    ax.pie(values, colors=colors, startangle=90, wedgeprops=dict(width=0.38, edgecolor="white"))
    ax.set(aspect="equal")
    fig.patch.set_alpha(0.0)
    return fig

# ---- Emotion palette for the 6-way NLP emotion vocabulary (Happy/Sad/Stress/
# Angry/Fear/Neutral) — kept separate from MOOD_STYLE, which colors the 5-point
# Amazing/Happy/Normal/Sad/Angry mood scale used for manual picks.
# Colors per spec: Neutral=gray, Happy=green, Angry=red, Sad=deep blue,
# Fear=dark purple, Stress=orange. ----
EMOTION_COLORS = {
    "Happy":   "#22C55E",   # green
    "Sad":     "#1E3A8A",   # deep blue
    "Stress":  "#F97316",   # orange
    "Angry":   "#EF4444",   # red
    "Fear":    "#6B21A8",   # dark purple
    "Neutral": "#6B7280",   # gray
}

# Colour emoji shown per emotion (rendered as real HTML/browser emoji in the
# legend chips, since headless matplotlib fonts can't guarantee colour glyphs).
EMOTION_EMOJI = {
    "Happy":   "😊",
    "Sad":     "😢",
    "Stress":  "😖",
    "Angry":   "😠",
    "Fear":    "😰",
    "Neutral": "😐",
}

def render_emotion_legend(labels):
    """Row of colour-coded emoji chips above the emotion bar chart — real
    browser-rendered colour emoji, one chip per emotion currently shown."""
    chips = "".join(
        f"<span class='pt-emo-chip' style='--c:{EMOTION_COLORS.get(l, MUTED)}'>"
        f"<span class='emo'>{EMOTION_EMOJI.get(l, '🔘')}</span>"
        f"<span class='lbl'>{l}</span></span>"
        for l in labels
    )
    st.markdown(f"<div class='pt-emo-legend'>{chips}</div>", unsafe_allow_html=True)

def _clean_emotion_label(raw: str) -> str:
    """nlp_pipeline.py stores emotion values as e.g. "Sad 😢" — the plain
    label plus its own emoji suffix. Strip that suffix so downstream lookups
    against EMOTION_COLORS / EMOTION_EMOJI (keyed by the plain word) match."""
    if not raw:
        return raw
    m = re.match(r"^[A-Za-z]+", raw.strip())
    return m.group(0) if m else raw.strip()

def donut_chart_with_legend(counts: dict, size=(5.2, 3.2)):
    """Mood Distribution donut with wedge-level percentages plus a side legend
    (colored dot + label + %), matching the reference dashboard mockup."""
    labels, values, colors = [], [], []
    for k in MOOD_LABELS:
        v = counts.get(k, 0)
        if v > 0:
            labels.append(k); values.append(v); colors.append(style_for(k)["color"])
    if not values:
        return None
    total = sum(values)
    fig, ax = plt.subplots(figsize=size)
    wedges, _texts, autotexts = ax.pie(
        values, colors=colors, startangle=90,
        wedgeprops=dict(width=0.38, edgecolor="white"),
        autopct=lambda p: f"{p:.0f}%" if p > 0 else "",
        pctdistance=0.82,
    )
    for t in autotexts:
        t.set_color("white"); t.set_fontsize(9); t.set_fontweight("bold")
    ax.set(aspect="equal")
    fig.patch.set_alpha(0.0)
    legend_labels = [f"{style_for(l)['emoji']} {l}   {v/total*100:.0f}%" for l, v in zip(labels, values)]
    ax.legend(wedges, legend_labels, loc="center left", bbox_to_anchor=(1.02, 0.5),
              frameon=False, fontsize=9, labelspacing=1.1)
    fig.tight_layout()
    return fig

def mood_trend_chart(trend: dict, size=(6.2, 3.2)):
    """Line chart with markers for the daily average mood score, styled like
    the reference dashboard (single accent-colored line, zero baseline)."""
    if not trend:
        return None
    dates = [str(d) for d in trend.keys()]
    values = list(trend.values())
    fig, ax = plt.subplots(figsize=size)
    ax.plot(dates, values, marker="o", color=ACCENT, linewidth=2.2, markersize=5,
             markerfacecolor=ACCENT, markeredgecolor="white")
    ax.axhline(0, color=BORDER, linewidth=1, linestyle="--")
    ax.set_ylim(-2.2, 2.2)
    ax.set_ylabel("Mood score", fontsize=9, color=MUTED)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.grid(axis="y", color=BORDER, alpha=0.6)
    ax.tick_params(axis="x", labelrotation=45, labelsize=8)
    ax.tick_params(axis="y", labelsize=8)
    fig.patch.set_alpha(0.0)
    fig.tight_layout()
    return fig

def emotion_bar_chart(emo_counts: dict, size=(6.4, 3.6), value_fmt=None):
    """Bar chart styled after the reference pill/capsule UI: slim bars with
    a fully rounded top, a soft vertical gradient sheen, no y-axis clutter,
    and the label (plus its emoji) directly underneath each bar in that
    emotion's own colour — using this app's own EMOTION_COLORS palette,
    not the rainbow colours from the reference image. Used both for whole-
    number journal-entry counts (Dashboard) and 0..1 model probability
    scores (Journal) — value_fmt auto-detects which, or pass your own."""
    from matplotlib.patches import FancyBboxPatch
    from matplotlib.colors import LinearSegmentedColormap, to_rgb
    import numpy as np

    labels = list(emo_counts.keys())
    values = list(emo_counts.values())
    colors = [EMOTION_COLORS.get(l, MUTED) for l in labels]
    maxv = max(values) if values else 1

    if value_fmt is None:
        if all(float(v).is_integer() for v in values):
            value_fmt = lambda v: str(int(v))
        else:
            value_fmt = lambda v: f"{v:.0%}"

    fig, ax = plt.subplots(figsize=size)
    bar_w = 0.46
    cap_r = bar_w / 2 * 0.95   # rounding radius -> fully-rounded pill top
    for i, (v, c) in enumerate(zip(values, colors)):
        x0 = i - bar_w / 2
        # Box is drawn starting below y=0 by the cap radius so the (also
        # rounded) bottom corner sits off-screen once clipped to ylim=0..,
        # leaving a flat base and a clean rounded/capsule top.
        rect = FancyBboxPatch(
            (x0, -cap_r), bar_w, v + cap_r,
            boxstyle=f"round,pad=0,rounding_size={cap_r:.4f}",
            linewidth=0, facecolor="none", zorder=2,
        )
        ax.add_patch(rect)
        light = tuple(min(1, ch + (1 - ch) * 0.6) for ch in to_rgb(c))
        cmap = LinearSegmentedColormap.from_list("grad", [light, c])
        grad = np.linspace(0, 1, 256).reshape(-1, 1)
        im = ax.imshow(grad, cmap=cmap, aspect="auto", origin="upper",
                        extent=(x0, x0 + bar_w, 0, v), zorder=2)
        im.set_clip_path(rect)
        ax.text(i, v + maxv * 0.05, value_fmt(v), ha="center", va="bottom",
                 fontsize=9.5, fontweight="bold", color=c, zorder=3)

    ax.set_xlim(-0.62, len(labels) - 0.38 if labels else 0.62)
    ax.set_ylim(0, maxv * 1.3 if maxv else 1)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels([f"{EMOTION_EMOJI.get(l, '')}  {l}" for l in labels],
                        fontsize=9.5, fontweight="700")
    for tick, c in zip(ax.get_xticklabels(), colors):
        tick.set_color(c)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.grid(axis="y", color=BORDER, alpha=0.5, linewidth=0.8, zorder=0)
    ax.set_yticks([])
    ax.tick_params(axis="x", length=0, pad=10)
    ax.set_facecolor("none")
    fig.patch.set_alpha(0.0)
    fig.tight_layout()
    return fig

def build_csv_export(history) -> bytes:
    """Flattens the (already filtered) mood log rows into a downloadable CSV."""
    buf = io.StringIO()
    writer = csv.writer(buf)
    writer.writerow(["Date", "Time", "Mood", "Emotion", "Compound Score", "Source", "Journal Entry"])
    for h in history:
        writer.writerow([
            h["mood_date"],
            h["created_at"].strftime("%H:%M:%S"),
            h["sentiment"] or "",
            h["emotion"] or "",
            h["compound_score"] if h["compound_score"] is not None else "",
            h["source"],
            (h.get("journal_text") or "").replace("\n", " ").strip(),
        ])
    return buf.getvalue().encode("utf-8")

def build_pdf_report(user, history, counts, trend, emo_counts) -> bytes:
    """Renders a multi-page PDF (summary + charts + activity table) covering
    exactly the currently filtered dashboard data, using matplotlib's PDF
    backend so no extra PDF dependency is needed."""
    buf = io.BytesIO()
    with PdfPages(buf) as pdf:
        # ---- Cover / summary page ----
        fig = plt.figure(figsize=(8.27, 11.69))  # A4
        fig.text(0.08, 0.94, f"{BRAND_NAME} Wellness Report", fontsize=20, fontweight="bold", color=PRIMARY)
        fig.text(0.08, 0.915, BRAND_TAGLINE, fontsize=11, color=MUTED)
        fig.text(0.08, 0.875, f"Employee: {user['username']} ({user['email']})", fontsize=11, color=INK)
        fig.text(0.08, 0.85, f"Generated: {datetime.now().strftime('%b %d, %Y %H:%M')}", fontsize=10, color=MUTED)
        fig.text(0.08, 0.825, f"Entries in report: {len(history)}", fontsize=10, color=MUTED)
        if history:
            dmin = min(h["mood_date"] for h in history)
            dmax = max(h["mood_date"] for h in history)
            fig.text(0.08, 0.80, f"Date range: {dmin} → {dmax}", fontsize=10, color=MUTED)
        fig.text(0.08, 0.76, "Mood breakdown", fontsize=13, fontweight="bold", color=INK)
        y = 0.73
        total = sum(counts.values()) or 1
        for k in MOOD_LABELS:
            v = counts.get(k, 0)
            if v:
                fig.text(0.10, y, f"{style_for(k)['emoji']}  {k}: {v}  ({v/total*100:.0f}%)", fontsize=10, color=INK)
                y -= 0.028
        pdf.savefig(fig); plt.close(fig)

        # ---- Chart pages (reuse the same chart functions as the dashboard) ----
        donut_fig = donut_chart_with_legend(counts, size=(7, 4.2))
        if donut_fig:
            donut_fig.suptitle("Mood Distribution", fontsize=13, fontweight="bold")
            pdf.savefig(donut_fig); plt.close(donut_fig)

        trend_fig = mood_trend_chart(trend, size=(7.5, 4))
        if trend_fig:
            trend_fig.suptitle("Mood Trend Over Time", fontsize=13, fontweight="bold")
            pdf.savefig(trend_fig); plt.close(trend_fig)

        if emo_counts:
            emo_fig = emotion_bar_chart(emo_counts, size=(7.5, 4))
            emo_fig.suptitle("Emotions Detected From Journal Entries", fontsize=13, fontweight="bold")
            pdf.savefig(emo_fig); plt.close(emo_fig)

        # ---- Activity table page(s) ----
        rows_per_page = 25
        chunk = history[:200]
        for i in range(0, max(len(chunk), 1), rows_per_page):
            page_rows = chunk[i:i + rows_per_page]
            if not page_rows:
                break
            fig, ax = plt.subplots(figsize=(8.27, 11.69))
            ax.axis("off")
            table_data = [["Date", "Time", "Mood", "Emotion", "Source"]]
            for h in page_rows:
                table_data.append([
                    str(h["mood_date"]), h["created_at"].strftime("%H:%M"),
                    h["sentiment"] or "", h["emotion"] or "—", h["source"],
                ])
            tbl = ax.table(cellText=table_data, loc="upper center", cellLoc="left")
            tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.5)
            ax.set_title("Recent Activity" if i == 0 else "Recent Activity (cont.)",
                          fontsize=12, fontweight="bold", loc="left")
            pdf.savefig(fig); plt.close(fig)
    buf.seek(0)
    return buf.getvalue()

def kpi_tile(icon, label, value, sub=None, color=PRIMARY):
    sub_html = f"<div class='kpi-sub'>{sub}</div>" if sub else ""
    st.markdown(
        f"<div class='pt-kpi' style='--kpi-color:{color}'>"
        f"<div class='kpi-top'><span class='kpi-label'>{label}</span>"
        f"<span class='kpi-icon'>{icon}</span></div>"
        f"<div class='kpi-value'>{value}</div>{sub_html}</div>",
        unsafe_allow_html=True,
    )

def section_header(icon, title, subtitle=None):
    sub_html = f"<span class='st'>· {subtitle}</span>" if subtitle else ""
    st.markdown(
        f"<div class='pt-section'><span class='ic'>{icon}</span>"
        f"<span class='tt'>{title}</span>{sub_html}</div>",
        unsafe_allow_html=True,
    )

def initials(name: str) -> str:
    parts = [p for p in re.split(r"\s+", name.strip()) if p]
    if not parts:
        return "?"
    if len(parts) == 1:
        return parts[0][:2].upper()
    return (parts[0][0] + parts[-1][0]).upper()

def page_header(title, subtitle, chip=None):
    chip_html = f"<div class='pt-header-chip'>{chip}</div>" if chip else ""
    st.markdown(
        f"<div class='pt-header'><div><h1>{title}</h1><p>{subtitle}</p></div>{chip_html}</div>",
        unsafe_allow_html=True,
    )

inject_css()

@st.cache_resource
def setup(): init_db()
setup()

if "page" not in st.session_state: st.session_state.page = "welcome"
if "show_auth_panel" not in st.session_state: st.session_state.show_auth_panel = False
if "auth_mode" not in st.session_state: st.session_state.auth_mode = "login"
if "token" not in st.session_state: st.session_state.token = None
if "email" not in st.session_state: st.session_state.email = None
if "chat_history" not in st.session_state: st.session_state.chat_history = []
if "cal_year" not in st.session_state: st.session_state.cal_year = date.today().year
if "cal_month" not in st.session_state: st.session_state.cal_month = date.today().month
if "today_mood_saved" not in st.session_state: st.session_state.today_mood_saved = False
if "nav" not in st.session_state: st.session_state.nav = "Home"

def goto_auth(mode): st.session_state.auth_mode = mode; st.rerun()

def valid_pw(pw):
    return len(pw) >= 8 and re.search(r"[A-Za-z]", pw) and re.search(r"[0-9]", pw)

# ═════════════════════════════════════════════════════════════════════════
# LOGGED-IN EXPERIENCE
# ═════════════════════════════════════════════════════════════════════════
if st.session_state.token:
    user = read_token(st.session_state.token)
    if user:
        role = user.get("role", "employee")
        headers = {"Authorization": f"Bearer {st.session_state.token}"}

        with st.sidebar:
            st.markdown(
                f"<div class='pt-logo-row'>"
                f"<div class='pt-logo-badge'>{BRAND_ICON}</div>"
                f"<div class='pt-logo-text'><div class='name'>{BRAND_NAME}</div>"
                f"<div class='tag'>{BRAND_TAGLINE}</div></div></div>",
                unsafe_allow_html=True,
            )
            st.markdown("<div class='pt-nav-label'>Navigate</div>", unsafe_allow_html=True)
            if role == "employee":
                nav_options = ["Home", "Journal", "Wellness Chat", "Face Scanner", "Dashboard"]
            else:
                nav_options = ["Analytics Dashboard"]
            st.session_state.nav = st.radio(
                "Navigate", nav_options,
                index=nav_options.index(st.session_state.nav) if st.session_state.nav in nav_options else 0,
                label_visibility="collapsed",
                format_func=lambda x: f"{NAV_ICONS.get(x, '•')}  {x}",
            )

            role_class = "pt-role-manager" if role == "manager" else "pt-role-employee"
            st.markdown(
                f"<div class='pt-profile-card'>"
                f"<div class='pt-avatar'>{initials(user['username'])}</div>"
                f"<div><div class='pt-profile-name'>{user['username']}</div>"
                f"<div class='pt-profile-meta'>{user['email']}</div>"
                f"<span class='pt-role-pill {role_class}'>{role}</span></div></div>",
                unsafe_allow_html=True,
            )
            st.write("")
            backend_status_chip()
            st.write("")
            if st.button("🚪  Log out", use_container_width=True):
                st.session_state.token = None
                st.session_state.page = "welcome"
                st.session_state.show_auth_panel = False
                st.rerun()

        greeting = "Good Morning" if datetime.now().hour < 12 else (
            "Good Afternoon" if datetime.now().hour < 18 else "Good Evening")
        now = datetime.now()

        if role == "employee":
            section = st.session_state.nav

            if section == "Home":
                page_header(f"{greeting}, {user['username']} 👋",
                            "Here's your personal wellness overview for today.",
                            chip=f"📅 {now.strftime('%b %d, %Y')}")

                history_all = get_user_mood_history(user["id"], limit=500)
                latest = history_all[0] if history_all else None
                today_count = sum(1 for h in history_all if h["mood_date"] == date.today())
                streak = 0
                day_ptr = date.today()
                day_set = {h["mood_date"] for h in history_all}
                while day_ptr in day_set:
                    streak += 1
                    day_ptr = date.fromordinal(day_ptr.toordinal() - 1)

                positive_count = sum(1 for h in history_all if h["sentiment"] in ("Amazing", "Happy"))
                overall_score = int(100 * positive_count / len(history_all)) if history_all else 0

                m1, m2, m3, m4 = st.columns(4)
                with m1:
                    if latest:
                        s = style_for(latest["sentiment"])
                        kpi_tile("🙂", "Current Mood", f"{s['emoji']} {latest['sentiment']}", color=s["color"])
                    else:
                        kpi_tile("🙂", "Current Mood", "—")
                with m2:
                    kpi_tile("📈", "Overall Score", f"{overall_score}%",
                             "Positive trend" if overall_score >= 50 else "Needs attention",
                             color=ACCENT if overall_score >= 50 else WARNING)
                with m3:
                    kpi_tile("✅", "Entries Today", today_count, color=PRIMARY)
                with m4:
                    kpi_tile("🔥", "Current Streak", f"{streak} days", color="#EA580C")

                st.write("")
                with st.container(border=True):
                    section_header("💭", "How Do You Feel Right Now?", now.strftime("%H:%M"))
                    cols = st.columns(len(MOOD_LABELS))
                    picked = st.session_state.get("picked_mood")
                    for col, label in zip(cols, MOOD_LABELS):
                        s = style_for(label)
                        with col:
                            selected = picked == label
                            border = f"2px solid {s['color']}" if selected else f"1px solid {BORDER}"
                            st.markdown(
                                f"<div style='text-align:center;padding:10px 4px;border-radius:12px;border:{border};background:{s['color']}0d'>"
                                f"<div style='font-size:30px'>{s['emoji']}</div>"
                                f"<div style='color:{s['color']};font-weight:700;font-size:12.5px'>{label}</div></div>",
                                unsafe_allow_html=True,
                            )
                            if st.button("Select", key=f"pick_{label}", use_container_width=True):
                                st.session_state.picked_mood = label
                                st.rerun()

                    st.write("")
                    confirm_col = st.columns([3, 1, 3])[1]
                    with confirm_col:
                        disabled = picked is None
                        if st.button("Save mood", type="primary", disabled=disabled,
                                     use_container_width=True):
                            save_manual_mood(user["id"], st.session_state.picked_mood)
                            st.session_state.today_mood_saved = True
                            st.session_state.picked_mood = None
                            st.rerun()

                    if st.session_state.today_mood_saved:
                        st.success("Today's mood saved!")
                        st.session_state.today_mood_saved = False

                st.write("")
                with st.container(border=True):
                    section_header("🗓️", "Your Mood Calendar")

                    nav_l, nav_mid, nav_r = st.columns([1, 3, 1])
                    if nav_l.button("← Prev", use_container_width=True):
                        m, y = st.session_state.cal_month - 1, st.session_state.cal_year
                        if m == 0: m, y = 12, y - 1
                        st.session_state.cal_month, st.session_state.cal_year = m, y
                        st.rerun()
                    if nav_r.button("Next →", use_container_width=True):
                        m, y = st.session_state.cal_month + 1, st.session_state.cal_year
                        if m == 13: m, y = 1, y + 1
                        st.session_state.cal_month, st.session_state.cal_year = m, y
                        st.rerun()
                    nav_mid.markdown(
                        f"<h4 style='text-align:center;margin:6px 0;color:{INK}'>{calendar.month_name[st.session_state.cal_month]} "
                        f"{st.session_state.cal_year}</h4>", unsafe_allow_html=True,
                    )

                    logs = get_mood_logs_for_month(user["id"], st.session_state.cal_year,
                                                   st.session_state.cal_month)
                    by_day = {row["mood_date"].day: row for row in logs}

                    weeks = calendar.Calendar(firstweekday=6).monthdayscalendar(
                        st.session_state.cal_year, st.session_state.cal_month
                    )
                    day_names = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
                    header_cols = st.columns(7)
                    for c, name in zip(header_cols, day_names):
                        c.markdown(f"<div style='text-align:center;color:{MUTED};font-weight:700;font-size:11.5px'>{name}</div>",
                                   unsafe_allow_html=True)

                    for week in weeks:
                        cols = st.columns(7)
                        for col, day_num in zip(cols, week):
                            if day_num == 0:
                                col.write("")
                                continue
                            entry = by_day.get(day_num)
                            s = style_for(entry["sentiment"] if entry else None)
                            time_label = entry["created_at"].strftime("%H:%M") if entry else ""
                            col.markdown(
                                f"<div class='pt-cal-cell' style='--c:{s['color']}22' title='{time_label}'>"
                                f"<div class='pt-cal-day'>{day_num}</div>"
                                f"<div class='pt-cal-emoji'>{s['emoji']}</div>"
                                f"<div class='pt-cal-time'>{time_label}</div></div>",
                                unsafe_allow_html=True,
                            )

                    legend = " &nbsp;·&nbsp; ".join(f"{style_for(l)['emoji']} {l}" for l in MOOD_LABELS)
                    st.markdown(f"<p style='color:{MUTED};font-size:12px;margin-top:10px'>{legend} &nbsp;·&nbsp; ⬜ No entry logged</p>",
                                unsafe_allow_html=True)

            elif section == "Journal":
                page_header("Journal", "Reflect on your day — analyzed automatically for sentiment and emotion.",
                            chip="🤖 NLP Engine")

                with st.container(border=True):
                    section_header("✍️", "Write an entry")
                    journal_text = st.text_area(
                        "Write about how you're feeling today", height=150,
                        label_visibility="collapsed",
                        placeholder="Your note here...",
                    )
                    if st.button("Analyze my entry", type="primary", use_container_width=True):
                        if not journal_text.strip():
                            st.warning("Write something first.")
                        else:
                            with st.spinner("Running NLP analysis…"):
                                try:
                                    resp = requests.post(
                                        f"{BACKEND_URL}/analyze-text",
                                        json={"text": journal_text},
                                        headers=headers, timeout=120,
                                    )
                                except requests.exceptions.RequestException as e:
                                    st.error(f"Could not reach backend: {e}"); resp = None
                            if resp is not None:
                                if resp.status_code != 200:
                                    st.error("Analysis failed.")
                                else:
                                    r = resp.json()
                                    save_mood_log(
                                        user["id"], r["final_sentiment"], r["final_emotion"],
                                        r["sentiment_scores"]["compound"], journal_text,
                                    )
                                    st.success(f"Saved! Sentiment: **{r['final_sentiment']}**, "
                                               f"Emotion: **{r['final_emotion']}**")
                                    render_emotion_legend(list(r["emotion_scores"].keys()))
                                    fig_j = emotion_bar_chart(r["emotion_scores"], size=(6.4, 3.2))
                                    st.pyplot(fig_j, use_container_width=True)
                                    top_emotion = max(r["emotion_scores"], key=r["emotion_scores"].get)
                                    render_wellness_suggestions(top_emotion, journal_text)

                st.write("")
                with st.container(border=True):
                    section_header("📎", "Or upload a file", "CSV or TXT")
                    uploaded = st.file_uploader("Choose a CSV or TXT file", type=["csv", "txt"],
                                                 label_visibility="collapsed")
                    if uploaded is not None and st.button("Run NLP Analysis on file", use_container_width=True):
                        files = {"file": (uploaded.name, uploaded.getvalue())}
                        with st.spinner("Running multilingual NLP pipeline…"):
                            try:
                                resp = requests.post(f"{BACKEND_URL}/analyze", files=files,
                                                     headers=headers, timeout=120)
                            except requests.exceptions.RequestException as e:
                                st.error(f"Could not reach backend: {e}"); resp = None
                        if resp is not None:
                            if resp.status_code != 200:
                                st.error("Analysis failed.")
                            else:
                                r = resp.json()
                                save_mood_log(
                                    user["id"], r["final_sentiment"], r["final_emotion"],
                                    r["sentiment_scores"]["compound"], r.get("cleaned_text", ""),
                                )
                                st.success(f"Saved! Sentiment: **{r['final_sentiment']}**, "
                                           f"Emotion: **{r['final_emotion']}**")
                                render_emotion_legend(list(r["emotion_scores"].keys()))
                                fig_j2 = emotion_bar_chart(r["emotion_scores"], size=(6.4, 3.2))
                                st.pyplot(fig_j2, use_container_width=True)
                                top_emotion = max(r["emotion_scores"], key=r["emotion_scores"].get)
                                render_wellness_suggestions(top_emotion, r.get("cleaned_text", ""))

                st.write("")
                section_header("🗂️", "Past entries")
                history = [h for h in get_user_mood_history(user["id"], limit=20)
                           if h["journal_text"]]
                if not history:
                    st.caption("No journal entries yet.")
                for h in history:
                    s = style_for(h["sentiment"])
                    with st.expander(
                        f"{s['emoji']} {h['sentiment']} — {h['created_at'].strftime('%Y-%m-%d %H:%M')}"
                    ):
                        st.write(h["journal_text"])

            elif section == "Wellness Chat":
                page_header("Wellness Chat", "A supportive space to talk about how you're feeling.",
                            chip="💬 Not a substitute for professional care")

                chat_box = st.container(height=450, border=True)
                with chat_box:
                    for turn in st.session_state.chat_history:
                        with st.chat_message(turn["role"]):
                            st.write(turn["content"])

                user_msg = st.chat_input("How are you feeling today?")
                if user_msg:
                    st.session_state.chat_history.append({"role": "user", "content": user_msg})
                    recent_history = st.session_state.chat_history[-10:-1]
                    try:
                        resp = requests.post(
                            f"{BACKEND_URL}/chat",
                            json={"message": user_msg, "history": recent_history},
                            headers=headers, timeout=60,
                        )
                        reply = resp.json()["reply"] if resp.status_code == 200 else \
                            "Sorry, I couldn't reach the wellness assistant right now."
                    except requests.exceptions.RequestException:
                        reply = "Sorry, I couldn't reach the wellness assistant right now."
                    st.session_state.chat_history.append({"role": "assistant", "content": reply})
                    st.rerun()

                if st.session_state.chat_history and st.button("Clear chat"):
                    st.session_state.chat_history = []
                    st.rerun()

            elif section == "Face Scanner":
                page_header("Live Face Scanner", "Analyze your current emotion from a webcam snapshot using DeepFace.",
                            chip="🤖 Computer Vision")

                with st.container(border=True):
                    img_file_buffer = st.camera_input("Take a picture")

                    if img_file_buffer is not None:
                        bytes_data = img_file_buffer.getvalue()
                        cv2_img = cv2.imdecode(np.frombuffer(bytes_data, np.uint8), cv2.IMREAD_COLOR)

                        with st.spinner("Analyzing face..."):
                            try:
                                results = DeepFace.analyze(
                                    cv2_img,
                                    actions=["emotion"],
                                    detector_backend="opencv",
                                    enforce_detection=False
                                )

                                for face in results:
                                    region = face["region"]
                                    emotion = face["dominant_emotion"]
                                    x, y, w, h = region["x"], region["y"], region["w"], region["h"]

                                    cv2.rectangle(cv2_img, (x, y), (x+w, y+h), (0, 255, 0), 2)
                                    cv2.putText(cv2_img, emotion.capitalize(), (x, y-10),
                                                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

                                st.image(cv2_img, channels="BGR", caption=f"Detected Emotion: {emotion.capitalize()}")

                                render_wellness_suggestions(
                                    FACE_TO_WELLNESS_EMOTION.get(emotion.lower(), "Neutral")
                                )

                                mapped_mood = "Normal"
                                if emotion.lower() in ["happy", "surprise"]:
                                    mapped_mood = "Happy"
                                elif emotion.lower() in ["sad", "fear"]:
                                    mapped_mood = "Sad"
                                elif emotion.lower() in ["angry", "disgust"]:
                                    mapped_mood = "Angry"

                                if st.button(f"Save as '{mapped_mood}'", type="primary"):
                                    save_manual_mood(user["id"], mapped_mood)
                                    st.success("Emotion logged successfully!")

                            except Exception as e:
                                st.error(f"Could not detect a face or analyze emotions. Error: {e}")

            elif section == "Dashboard":
                page_header("My Dashboard", "Your personal wellness analytics, at a glance.",
                            chip=f"📅 {now.strftime('%b %d, %Y')}")

                # Pull a larger window once; all filters below operate on this
                # in-memory list so date/mood/emotion/search filtering is instant
                # and every KPI, chart, table, and export always uses the SAME
                # rows — i.e. real backend/DB data, never separately-mocked numbers.
                full_history = get_user_mood_history(user["id"], limit=500)
                if not full_history:
                    st.markdown(
                        "<div class='pt-empty-state'><div class='ic'>🌱</div>"
                        "<div class='tt'>No entries yet</div>"
                        "<div class='sub'>Pick a mood on Home or write a journal entry to see your dashboard.</div>"
                        "</div>",
                        unsafe_allow_html=True,
                    )
                else:
                    # ── KPI strip ──────────────────────────────────────────
                    total_entries = len(full_history)
                    avg_score = sum(MOOD_TO_NUM.get(h["sentiment"], 0) for h in full_history) / total_entries
                    mood_tally = {}
                    for h in full_history:
                        if h["sentiment"]:
                            mood_tally[h["sentiment"]] = mood_tally.get(h["sentiment"], 0) + 1
                    top_mood = max(mood_tally, key=mood_tally.get) if mood_tally else "—"

                    # Current streak of consecutive logged days, ending today
                    # (or still "live" if the last entry was yesterday and
                    # today hasn't been logged yet).
                    logged_days = sorted({h["mood_date"] for h in full_history}, reverse=True)
                    streak = 0
                    if logged_days:
                        expected = date.today()
                        if logged_days[0] == expected - timedelta(days=1):
                            expected = logged_days[0]
                        for d in logged_days:
                            if d == expected:
                                streak += 1
                                expected = expected - timedelta(days=1)
                            else:
                                break

                    k1, k2, k3, k4 = st.columns(4)
                    with k1:
                        kpi_tile("🗒️", "Total Entries", total_entries, color=PRIMARY)
                    with k2:
                        s_top = style_for(top_mood)
                        kpi_tile("🏆", "Most Frequent Mood", f"{s_top['emoji']} {top_mood}", color=s_top["color"])
                    with k3:
                        kpi_tile("📊", "Avg. Mood Score", f"{avg_score:+.2f}", "Scale: −2 to +2",
                                  color=ACCENT if avg_score >= 0 else WARNING)
                    with k4:
                        kpi_tile("🔥", "Current Streak", f"{streak} day{'s' if streak != 1 else ''}",
                                  color=PRIMARY if streak else MUTED)

                    st.write("")

                    # ── Filters toolbar (collapsed by default) ──────────────
                    min_d = min(h["mood_date"] for h in full_history)
                    max_d = max(h["mood_date"] for h in full_history)
                    mood_opts = [m for m in MOOD_LABELS if any(h["sentiment"] == m for h in full_history)]
                    emo_opts = sorted({h["emotion"] for h in full_history if h["emotion"]})

                    with st.expander("🔍  Filters — date range, mood, emotion, search", expanded=False):
                        fc1, fc2, fc3 = st.columns([1.2, 1.3, 1.3])
                        with fc1:
                            date_range = st.date_input(
                                "Date range", value=(min_d, max_d),
                                min_value=min_d, max_value=max_d, key="dash_date_range",
                            )
                        with fc2:
                            mood_filter = st.multiselect("Mood", mood_opts, default=mood_opts,
                                                          key="dash_mood_filter")
                        with fc3:
                            emo_filter = st.multiselect("Emotion", emo_opts, default=emo_opts,
                                                         key="dash_emo_filter")
                        search_q = st.text_input(
                            "🔎 Search journal entries",
                            placeholder="Search the text of your journal entries...",
                            key="dash_search",
                        )

                    if isinstance(date_range, tuple) and len(date_range) == 2:
                        start_d, end_d = date_range
                    else:
                        start_d, end_d = min_d, max_d

                    active_filters = []
                    if (start_d, end_d) != (min_d, max_d): active_filters.append("date range")
                    if mood_filter and len(mood_filter) != len(mood_opts): active_filters.append("mood")
                    if emo_filter and len(emo_filter) != len(emo_opts): active_filters.append("emotion")
                    if search_q and search_q.strip(): active_filters.append("search")
                    if active_filters:
                        st.markdown(
                            f"<span class='pt-filter-chip'>🔎 Filtering by: {', '.join(active_filters)}</span>",
                            unsafe_allow_html=True,
                        )

                    def _matches_filters(h):
                        if not (start_d <= h["mood_date"] <= end_d):
                            return False
                        if mood_filter and h["sentiment"] not in mood_filter:
                            return False
                        if emo_filter and len(emo_filter) != len(emo_opts):
                            if h["emotion"] not in emo_filter:
                                return False
                        if search_q and search_q.strip():
                            if search_q.strip().lower() not in (h.get("journal_text") or "").lower():
                                return False
                        return True

                    history = [h for h in full_history if _matches_filters(h)]

                    st.write("")

                    if not history:
                        st.markdown(
                            "<div class='pt-empty-state'><div class='ic'>🔍</div>"
                            "<div class='tt'>No entries match your filters</div>"
                            "<div class='sub'>Try widening the date range or clearing a filter.</div></div>",
                            unsafe_allow_html=True,
                        )
                    else:
                        counts = {label: 0 for label in MOOD_LABELS}
                        for h in history:
                            if h["sentiment"] in counts:
                                counts[h["sentiment"]] += 1

                        by_date = {}
                        for h in history:
                            d = h["mood_date"]
                            by_date.setdefault(d, []).append(MOOD_TO_NUM.get(h["sentiment"], 0))
                        trend = {d: sum(v) / len(v) for d, v in sorted(by_date.items())}

                        emo_counts = {}
                        for h in history:
                            if h["source"] == "nlp" and h["emotion"]:
                                # nlp_pipeline.py stores the emotion as e.g. "Sad 😢"
                                # (label + its own emoji suffix) — strip that so it
                                # groups and looks up cleanly against EMOTION_COLORS.
                                clean = _clean_emotion_label(h["emotion"])
                                emo_counts[clean] = emo_counts.get(clean, 0) + 1

                        tab_overview, tab_activity = st.tabs(["📊  Overview", "🕒  Activity & Export"])

                        with tab_overview:
                            c1, c2 = st.columns(2)
                            with c1:
                                with st.container(border=True):
                                    st.markdown("<span class='pt-dash-card-marker'></span>", unsafe_allow_html=True)
                                    section_header("🍩", "Mood Distribution")
                                    fig = donut_chart_with_legend(counts)
                                    if fig: st.pyplot(fig, use_container_width=True)
                                    else: st.caption("No mood data for this selection.")
                            with c2:
                                with st.container(border=True):
                                    st.markdown("<span class='pt-dash-card-marker'></span>", unsafe_allow_html=True)
                                    section_header("📈", "Mood Trend Over Time")
                                    fig2 = mood_trend_chart(trend)
                                    if fig2: st.pyplot(fig2, use_container_width=True)
                                    else: st.caption("Not enough data to draw a trend.")

                            st.write("")
                            with st.container(border=True):
                                st.markdown("<span class='pt-emo-card-marker'></span>", unsafe_allow_html=True)
                                section_header("🎭", "Emotions Detected From Journal Entries",
                                               "Colour-coded by emotion")
                                if emo_counts:
                                    render_emotion_legend(list(emo_counts.keys()))
                                    fig3 = emotion_bar_chart(emo_counts)
                                    st.pyplot(fig3, use_container_width=True)
                                else:
                                    st.caption("No journal-based emotion data yet.")

                        with tab_activity:
                            with st.container(border=True):
                                st.markdown("<span class='pt-dash-card-marker'></span>", unsafe_allow_html=True)
                                section_header("🕒", "Recent Activity",
                                               f"{len(history)} entr{'y' if len(history) == 1 else 'ies'} matching filters")
                                table_rows = [{
                                    "Date": h["mood_date"], "Time": h["created_at"].strftime("%H:%M"),
                                    "Mood": f"{style_for(h['sentiment'])['emoji']} {h['sentiment']}",
                                    "Emotion": h["emotion"] or "—",
                                    "Source": h["source"],
                                    "Journal": ((h["journal_text"][:80] + "…")
                                                if h.get("journal_text") and len(h["journal_text"]) > 80
                                                else (h.get("journal_text") or "")),
                                } for h in history[:50]]
                                st.dataframe(table_rows, use_container_width=True, hide_index=True)

                            st.write("")
                            with st.container(border=True):
                                st.markdown("<span class='pt-dash-card-marker'></span>", unsafe_allow_html=True)
                                section_header("⬇️", "Export", "Download exactly what's shown above")
                                e1, e2 = st.columns(2)
                                with e1:
                                    st.download_button(
                                        "⬇️ Download CSV", data=build_csv_export(history),
                                        file_name=f"mood_history_{user['username']}_{now.strftime('%Y%m%d')}.csv",
                                        mime="text/csv", use_container_width=True,
                                    )
                                with e2:
                                    st.download_button(
                                        "🧾 Download PDF Report",
                                        data=build_pdf_report(user, history, counts, trend, emo_counts),
                                        file_name=f"wellness_report_{user['username']}_{now.strftime('%Y%m%d')}.pdf",
                                        mime="application/pdf", use_container_width=True,
                                    )

        else:
            # ── Manager / HR Analytics Dashboard ──────────────────────────
            page_header("Employee Wellness Analytics", "Organization-wide sentiment insights and mood trends.",
                        chip=f"📅 {now.strftime('%b %d, %Y')}")

            latest = get_latest_mood_per_employee()
            if not latest:
                st.info("No employee entries yet.")
            else:
                total_employees = len(latest)
                at_risk = sum(1 for row in latest if row["sentiment"] in ("Sad", "Angry"))
                avg_score = sum(MOOD_TO_NUM.get(row["sentiment"], 0) for row in latest) / total_employees
                mood_counts = {}
                for row in latest:
                    mood_counts[row["sentiment"]] = mood_counts.get(row["sentiment"], 0) + 1
                top_mood = max(mood_counts, key=mood_counts.get) if mood_counts else "—"

                k1, k2, k3, k4 = st.columns(4)
                with k1:
                    kpi_tile("👥", "Employees Tracked", total_employees, color=PRIMARY)
                with k2:
                    kpi_tile("⚠️", "At-Risk (Sad/Angry)", at_risk,
                             f"{int(100*at_risk/total_employees)}% of team" if total_employees else None,
                             color=DANGER if at_risk > 0 else ACCENT)
                with k3:
                    kpi_tile("📊", "Avg. Team Mood Score", f"{avg_score:+.2f}",
                             "Scale: -2 to +2", color=ACCENT if avg_score >= 0 else WARNING)
                with k4:
                    s = style_for(top_mood)
                    kpi_tile("🏆", "Most Common Mood", f"{s['emoji']} {top_mood}", color=s["color"])

                st.write("")
                with st.container(border=True):
                    section_header("🧾", "Employee Snapshot", "Latest mood per employee")
                    table_rows = [{
                        "Employee": row["username"],
                        "Email": row["email"],
                        "Date": row["mood_date"],
                        "Time": row["created_at"].strftime("%H:%M"),
                        "Mood": f"{style_for(row['sentiment'])['emoji']} {row['sentiment']}",
                        "Emotion": row["emotion"],
                    } for row in latest]
                    st.dataframe(table_rows, use_container_width=True, hide_index=True)

                st.write("")
                with st.container(border=True):
                    section_header("📈", "Team Mood Trend", "Last 30 days")
                    history = get_all_employee_mood_logs(limit_days=30)
                    if not history:
                        st.info("Not enough data yet to draw a trend chart.")
                    else:
                        by_date = {}
                        for row in history:
                            d = row["mood_date"]
                            by_date.setdefault(d, []).append(MOOD_TO_NUM.get(row["sentiment"], 0))
                        trend = {str(d): sum(v) / len(v) for d, v in sorted(by_date.items())}
                        st.line_chart(trend)
                        st.caption("Average mood score per day across all employees "
                                   "(2 = Amazing, 1 = Happy, 0 = Normal, -1 = Sad, -2 = Angry)")

        st.markdown(f"<div class='pt-footer'>{BRAND_NAME} · {BRAND_TAGLINE} &nbsp;·&nbsp; © {now.year}</div>",
                    unsafe_allow_html=True)
        st.stop()
    st.session_state.token = None

# ═════════════════════════════════════════════════════════════════════════
# PUBLIC LANDING PAGE
# ═════════════════════════════════════════════════════════════════════════
if st.session_state.page == "welcome":

    nav_l, nav_r = st.columns([3, 1])
    with nav_l:
        st.markdown(
            f"<div class='pt-navbar'>"
            f"<div style='display:flex;align-items:center;gap:10px'>"
            f"<div class='pt-logo-badge'>{BRAND_ICON}</div>"
            f"<div><div style='font-weight:800;font-size:17px;color:{INK}'>{BRAND_NAME}</div>"
            f"<div style='font-size:10.5px;font-weight:700;letter-spacing:.04em;text-transform:uppercase;color:{MUTED}'>{BRAND_TAGLINE}</div></div>"
            f"</div></div>",
            unsafe_allow_html=True,
        )
    with nav_r:
        st.write("")
        backend_status_chip()

    if not st.session_state.show_auth_panel:
        st.markdown(
            f"<div class='pt-hero'>"
            f"<span class='pt-hero-badge'>🤖 AI-POWERED WELLNESS INTELLIGENCE</span>"
            f"<h1>Employee Wellness Management &amp; Analytics Platform</h1>"
            f"<p>Track mood, sentiment, and emotional wellbeing across your organization — "
            f"from daily check-ins and journaling to AI-driven text and facial emotion analysis, "
            f"all rolled up into real-time dashboards for HR and managers.</p>"
            f"</div>",
            unsafe_allow_html=True,
        )

        st.markdown(
            f"""
            <div class='pt-feature-grid'>
                <div class='pt-feature-card'>
                    <div class='fic' style='background:{PRIMARY_LIGHT}'>😊</div>
                    <h4>Daily Mood Check-ins</h4>
                    <p>Quick one-tap mood logging with a calendar heatmap of emotional history.</p>
                </div>
                <div class='pt-feature-card'>
                    <div class='fic' style='background:{ACCENT_LIGHT}'>🧠</div>
                    <h4>AI Sentiment &amp; Emotion NLP</h4>
                    <p>Multilingual text analysis detects sentiment and emotion from journals and reports.</p>
                </div>
                <div class='pt-feature-card'>
                    <div class='fic' style='background:#FEF3C7'>📷</div>
                    <h4>Facial Emotion Scanning</h4>
                    <p>Optional webcam-based emotion detection powered by computer vision.</p>
                </div>
                <div class='pt-feature-card'>
                    <div class='fic' style='background:#E0E7FF'>📊</div>
                    <h4>Manager Analytics Dashboard</h4>
                    <p>Org-wide sentiment trends, at-risk flags, and team mood scoring for HR.</p>
                </div>
            </div>
            <div class='pt-stats-strip'>
                <div class='pt-stat'><div class='num'>5</div><div class='lbl'>Mood States Tracked</div></div>
                <div class='pt-stat'><div class='num'>24/7</div><div class='lbl'>Wellness Chat Support</div></div>
                <div class='pt-stat'><div class='num'>Multi</div><div class='lbl'>Language NLP</div></div>
                <div class='pt-stat'><div class='num'>100%</div><div class='lbl'>Private &amp; Secure</div></div>
            </div>
            """,
            unsafe_allow_html=True,
        )

        st.write("")
        cta_col = st.columns([2, 1.4, 2])[1]
        with cta_col:
            if st.button("Get Started →", type="primary", use_container_width=True):
                st.session_state.show_auth_panel = True
                st.rerun()
        st.markdown(f"<div class='pt-footer'>{BRAND_NAME} · {BRAND_TAGLINE} &nbsp;·&nbsp; © {datetime.now().year}</div>",
                    unsafe_allow_html=True)
        st.stop()

    left, right = st.columns([1.05, 1])

    with left:
        st.markdown(
            f"""
            <div class='pt-pitch-card'>
                <h2>Employee Wellness Management &amp; Analytics</h2>
                <p>One platform to understand how your people are really feeling — and act on it early.</p>
                <div class='pt-pitch-item'><span class='ic'>📈</span>
                    <div class='tx'><b>Real-time Analytics</b><span>Live dashboards for individuals and HR teams</span></div></div>
                <div class='pt-pitch-item'><span class='ic'>🧠</span>
                    <div class='tx'><b>AI-Powered Insights</b><span>NLP sentiment &amp; emotion detection, multilingual</span></div></div>
                <div class='pt-pitch-item'><span class='ic'>🔒</span>
                    <div class='tx'><b>Private &amp; Secure</b><span>JWT-protected access, encrypted credentials</span></div></div>
                <div class='pt-pitch-item'><span class='ic'>💬</span>
                    <div class='tx'><b>Always-on Support</b><span>Wellness chat assistant whenever it's needed</span></div></div>
            </div>
            """,
            unsafe_allow_html=True,
        )

    with right:
        # Real container (not raw div-wrapping) so the card styling actually
        # wraps the form — a hidden marker lets the CSS find this exact box.
        with st.container(border=True):
            st.markdown('<span class="pt-auth-marker"></span>', unsafe_allow_html=True)
            mode = st.session_state.auth_mode

            if mode == "login":
                st.markdown(f"<h2 style='margin:6px 0 2px 0;font-weight:800;color:{AUTH_INK}'>Welcome Back!</h2>", unsafe_allow_html=True)
                st.caption("Login to your wellness dashboard")
                with st.form("login"):
                    email = st.text_input("Email", placeholder="Enter your email")
                    pw = st.text_input("Password", type="password", placeholder="Enter your password")
                    go = st.form_submit_button("Login", type="primary", use_container_width=True)
                if go:
                    u = get_user(email.strip().lower())
                    if not u or not check_pw(pw, u["password_hash"]):
                        st.error("Invalid email or password.")
                    elif not u["is_verified"]:
                        st.warning("Verify your email first.")
                        st.session_state.email = u["email"]; goto_auth("verify")
                    else:
                        st.session_state.token = make_token(u)
                        st.rerun()
                c1, c2 = st.columns(2)
                with c1:
                    st.markdown('<div class="pt-link-btn">', unsafe_allow_html=True)
                    if st.button("Sign up", use_container_width=True): goto_auth("signup")
                    st.markdown('</div>', unsafe_allow_html=True)
                with c2:
                    st.markdown('<div class="pt-link-btn">', unsafe_allow_html=True)
                    if st.button("Forgot password?", use_container_width=True): goto_auth("forgot")
                    st.markdown('</div>', unsafe_allow_html=True)

            elif mode == "signup":
                st.markdown(f"<h2 style='margin:6px 0 2px 0;font-weight:800;color:{AUTH_INK}'>Hello!</h2>", unsafe_allow_html=True)
                st.caption("Sign up to get started")
                with st.form("signup"):
                    username = st.text_input("Full Name", placeholder="Enter your full name")
                    email = st.text_input("Email", placeholder="Enter your email")
                    pw = st.text_input("Password", type="password", placeholder="Create password")
                    role_label = st.radio("I am signing up as a:", ["Employee", "Manager"], horizontal=True)
                    go = st.form_submit_button("Create Account", type="primary", use_container_width=True)
                if go:
                    email = email.strip().lower()
                    role = "manager" if role_label == "Manager" else "employee"
                    if len(username) < 3:
                        st.error("Username too short.")
                    elif not valid_pw(pw):
                        st.error("Password needs 8+ chars, letters and numbers.")
                    elif username_taken(username) or get_user(email):
                        st.error("Username or email already in use.")
                    else:
                        create_user(username, email, pw, role=role)
                        code = new_otp(); save_otp(email, code, "signup")
                        ok, msg = send_otp(email, code, "signup")
                        if ok:
                            st.session_state.email = email
                            st.success("Check your email for the code.")
                            goto_auth("verify")
                        else:
                            st.error(f"Email failed: {msg}")
                st.markdown('<div class="pt-link-btn" style="text-align:center">', unsafe_allow_html=True)
                if st.button("Already have an account? Log in"): goto_auth("login")
                st.markdown('</div>', unsafe_allow_html=True)

            elif mode == "verify":
                email = st.session_state.email
                st.markdown(f"<h2 style='margin:6px 0 2px 0;font-weight:800;color:{AUTH_INK}'>Verify OTP</h2>", unsafe_allow_html=True)
                st.caption(f"We have sent a 6-digit code to {email}")
                with st.form("verify"):
                    code = st.text_input("Code", max_chars=6, placeholder="Enter 6-digit code")
                    go = st.form_submit_button("Verify OTP", type="primary", use_container_width=True)
                if go:
                    if check_otp(email, code.strip(), "signup"):
                        verify_user(email)
                        st.success("Verified! Please log in.")
                        goto_auth("login")
                    else:
                        st.error("Invalid or expired code.")
                st.markdown('<div class="pt-link-btn" style="text-align:center">', unsafe_allow_html=True)
                if st.button("← Back to login"): goto_auth("login")
                st.markdown('</div>', unsafe_allow_html=True)

            elif mode == "forgot":
                st.markdown(f"<h2 style='margin:6px 0 2px 0;font-weight:800;color:{AUTH_INK}'>🔑 Forgot Password</h2>", unsafe_allow_html=True)
                st.caption("We'll email you a reset code")
                with st.form("forgot"):
                    email = st.text_input("Your account email")
                    go = st.form_submit_button("Send reset code", type="primary", use_container_width=True)
                if go:
                    email = email.strip().lower()
                    if get_user(email):
                        code = new_otp(); save_otp(email, code, "password_reset")
                        send_otp(email, code, "password_reset")
                    st.session_state.email = email
                    st.info("If that email exists, a code was sent.")
                    goto_auth("reset")
                st.markdown('<div class="pt-link-btn" style="text-align:center">', unsafe_allow_html=True)
                if st.button("← Back to login"): goto_auth("login")
                st.markdown('</div>', unsafe_allow_html=True)

            elif mode == "reset":
                email = st.session_state.email
                st.markdown(f"<h2 style='margin:6px 0 2px 0;font-weight:800;color:{AUTH_INK}'>🔄 Reset Password</h2>", unsafe_allow_html=True)
                st.caption("Enter the code and choose a new password")
                with st.form("reset"):
                    code = st.text_input("Reset code", max_chars=6)
                    pw = st.text_input("New password", type="password")
                    go = st.form_submit_button("Reset", type="primary", use_container_width=True)
                if go:
                    if not valid_pw(pw):
                        st.error("Password needs 8+ chars, letters and numbers.")
                    elif not check_otp(email, code.strip(), "password_reset"):
                        st.error("Invalid or expired code.")
                    else:
                        set_password(email, pw)
                        st.success("Password reset. Please log in.")
                        goto_auth("login")
                st.markdown('<div class="pt-link-btn" style="text-align:center">', unsafe_allow_html=True)
                if st.button("← Back to login"): goto_auth("login")
                st.markdown('</div>', unsafe_allow_html=True)

        st.write("")
        st.markdown('<div class="pt-link-btn" style="text-align:center">', unsafe_allow_html=True)
        if st.button("← Back to home", use_container_width=True):
            st.session_state.show_auth_panel = False
            st.session_state.auth_mode = "login"
            st.rerun()
        st.markdown('</div>', unsafe_allow_html=True)

    st.stop()


## FastAPI backend (JWT-protected file upload)

This backend exposes a `/upload` endpoint that only accepts `.csv` or `.txt` files, verifies the same JWT issued at Streamlit login, and returns a preview (columns + first rows for CSV, first lines for TXT).


## Multilingual NLP pipeline module

Language detection, text cleaning, Telugu/Kannada stopword filtering, translation to English, lemmatization, VADER sentiment, and keyword-based emotion detection — imported by `backend.py`'s `/analyze` endpoint.


In [ ]:
%%writefile nlp_pipeline.py

import re
import ftfy
import emoji
import spacy
import torch
import stopwordsiso
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline as hf_pipeline,
)
from langdetect import detect, DetectorFactory
from deep_translator import GoogleTranslator
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

DetectorFactory.seed = 0

_nlp = None
_vader = None
_qwen_model = None
_qwen_tokenizer = None
_bert_emotion_pipeline = None

QWEN_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
BERT_EMOTION_MODEL_NAME = "bhadresh-savani/bert-base-go-emotion"

LANGUAGE_NAMES = {
    "te": "Telugu", "kn": "Kannada", "en": "English", "ta": "Tamil",
    "hi": "Hindi", "ml": "Malayalam", "mr": "Marathi", "bn": "Bengali", "gu": "Gujarati",
    "fr": "French", "de": "German", "es": "Spanish", "pt": "Portuguese",
    "ar": "Arabic", "zh": "Chinese", "ja": "Japanese", "ko": "Korean", "ru": "Russian",
}

def _get_stopwords(language_code: str) -> set:
    if stopwordsiso.has_lang(language_code):
        return stopwordsiso.stopwords(language_code)
    return set()

EMOTION_LABELS = ["Happy", "Sad", "Stress", "Angry", "Fear", "Neutral"]

EMOTION_EMOJI = {
    "Happy": "\U0001F60A", "Sad": "\U0001F622", "Stress": "\U0001F62B",
    "Angry": "\U0001F621", "Fear": "\U0001F628", "Neutral": "\U0001F610",
}

GOEMOTIONS_TO_APP_LABEL = {
    "joy": "Happy", "amusement": "Happy", "excitement": "Happy",
    "love": "Happy", "gratitude": "Happy", "optimism": "Happy",
    "relief": "Happy", "pride": "Happy", "admiration": "Happy",
    "approval": "Happy", "caring": "Happy",
    "sadness": "Sad", "disappointment": "Sad", "grief": "Sad",
    "remorse": "Sad",
    "nervousness": "Stress", "embarrassment": "Stress",
    "confusion": "Stress",
    "anger": "Angry", "annoyance": "Angry", "disgust": "Angry",
    "disapproval": "Angry",
    "fear": "Fear",
    "neutral": "Neutral", "realization": "Neutral", "surprise": "Neutral",
    "curiosity": "Neutral", "desire": "Neutral",
}

def _get_nlp():
    global _nlp
    if _nlp is None:
        _nlp = spacy.load("xx_sent_ud_sm")
    return _nlp

def _get_vader():
    global _vader
    if _vader is None:
        _vader = SentimentIntensityAnalyzer()
    return _vader

def _get_qwen():
    global _qwen_model, _qwen_tokenizer
    if _qwen_model is None:
        _qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
        _qwen_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_NAME,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )
    return _qwen_model, _qwen_tokenizer

def _get_bert_emotion_pipeline():
    global _bert_emotion_pipeline
    if _bert_emotion_pipeline is None:
        _bert_emotion_pipeline = hf_pipeline(
            "text-classification",
            model=BERT_EMOTION_MODEL_NAME,
            top_k=None,
            device=0 if torch.cuda.is_available() else -1,
        )
    return _bert_emotion_pipeline

def _bert_emotion(text: str) -> dict:
    classifier = _get_bert_emotion_pipeline()
    if not text.strip():
        text = "(empty feedback)"
    raw_predictions = classifier(text, truncation=True)[0]
    app_scores = {label: 0.0 for label in EMOTION_LABELS}
    for pred in raw_predictions:
        goemotion_label = pred["label"].lower()
        app_label = GOEMOTIONS_TO_APP_LABEL.get(goemotion_label, "Neutral")
        app_scores[app_label] += pred["score"]
    total = sum(app_scores.values()) or 1.0
    app_scores = {label: round(score / total, 4) for label, score in app_scores.items()}
    final_emotion = max(app_scores, key=app_scores.get)
    return {"emotion": final_emotion, "scores": app_scores}

def process_employee_feedback(text: str) -> dict:
    nlp = _get_nlp()
    vader = _get_vader()
    normalized_text = ftfy.fix_text(text)
    try:
        language = detect(normalized_text)
    except Exception:
        language = "unknown"
    detected_language = LANGUAGE_NAMES.get(language, "Other / Unknown")
    emoji_list = [ch for ch in normalized_text if ch in emoji.EMOJI_DATA]
    cleaned_text = re.sub(r"https?://\S+|www\.\S+", " ", normalized_text)
    cleaned_text = re.sub(r"\S+@\S+", " ", cleaned_text)
    cleaned_text = re.sub(r"@\w+|#\w+", " ", cleaned_text)
    cleaned_text = emoji.replace_emoji(cleaned_text, replace="")
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()
    doc = nlp(cleaned_text)
    sentences = [s.text.strip() for s in doc.sents if s.text.strip()]
    original_tokens = [t.text for t in doc if not t.is_space]
    clean_tokens = [t.text for t in doc if not t.is_punct and not t.is_space and not t.like_num]
    selected_stopwords = _get_stopwords(language)
    filtered_tokens = [t for t in clean_tokens if t.lower() not in selected_stopwords]
    final_preprocessed_text = " ".join(filtered_tokens)
    try:
        translated_text = GoogleTranslator(source="auto", target="en").translate(final_preprocessed_text)
    except Exception as error:
        translated_text = f"Translation failed: {error}"
    english_doc = nlp(translated_text)
    lemmas = [t.lemma_ if t.lemma_ else t.text for t in english_doc if not t.is_space]
    lemmatized_text = " ".join(lemmas)
    sentiment_scores = vader.polarity_scores(translated_text)
    compound_score = sentiment_scores["compound"]
    if compound_score >= 0.05:
        final_sentiment = "Positive \U0001F60A"
    elif compound_score <= -0.05:
        final_sentiment = "Negative \U0001F614"
    else:
        final_sentiment = "Neutral \U0001F610"
    bert_result = _bert_emotion(translated_text)
    emotion_scores = bert_result["scores"]
    final_emotion_label = bert_result["emotion"]
    final_emotion = f"{final_emotion_label} {EMOTION_EMOJI.get(final_emotion_label, '')}"
    return {
        "language_code": language,
        "detected_language": detected_language,
        "normalized_text": normalized_text,
        "cleaned_text": cleaned_text,
        "sentences": sentences,
        "original_tokens": original_tokens,
        "filtered_tokens": filtered_tokens,
        "emoji_list": emoji_list,
        "final_preprocessed_text": final_preprocessed_text,
        "translated_text": translated_text,
        "lemmatized_text": lemmatized_text,
        "sentiment_scores": sentiment_scores,
        "final_sentiment": final_sentiment,
        "emotion_scores": emotion_scores,
        "final_emotion": final_emotion,
    }

CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "want to die", "self harm",
    "self-harm", "hurt myself", "not worth living", "no reason to live",
]
CRISIS_MESSAGE = (
    "I'm really glad you reached out, and I want to make sure you get support "
    "beyond what I can offer here. If you're in immediate danger, please contact "
    "your local emergency number right now. You can also reach a crisis line: "
    "in India, AASRA is available at +91-9820466726 (24/7). If you're outside "
    "India, please look up a local crisis helpline or talk to a trusted person "
    "or your HR/EAP contact. You don't have to go through this alone."
)
WELLNESS_SYSTEM_PROMPT = (
    "You are a supportive workplace wellness assistant for employees. "
    "Your role is to listen, validate feelings, and offer general, gentle "
    "coping suggestions (like breathing exercises, taking a short break, "
    "or talking to a trusted colleague or manager). "
    "You are NOT a therapist or doctor: never diagnose any condition, never "
    "claim expertise you don't have, and never give medical or medication "
    "advice. If the employee describes something serious (ongoing crisis, "
    "self-harm, harming others), gently encourage them to contact a mental "
    "health professional, their HR/EAP program, or a crisis helpline. "
    "Keep replies short (2-4 sentences), warm, and non-judgmental. "
    "Avoid clinical labels and avoid being preachy or repetitive."
)

def _contains_crisis_language(text: str) -> bool:
    lowered = text.lower()
    return any(kw in lowered for kw in CRISIS_KEYWORDS)

def wellness_chat_reply(message: str, history: list[dict] | None = None) -> dict:
    if _contains_crisis_language(message):
        return {"reply": CRISIS_MESSAGE, "flagged": True}
    model, tokenizer = _get_qwen()
    messages = [{"role": "system", "content": WELLNESS_SYSTEM_PROMPT}]
    for turn in (history or []):
        if turn.get("role") in ("user", "assistant") and turn.get("content"):
            messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({"role": "user", "content": message})
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(generated, skip_special_tokens=True).strip()
    if not reply:
        reply = "I'm here and listening — could you tell me a bit more about how you're feeling?"
    return {"reply": reply, "flagged": False}

Writing nlp_pipeline.py


## Backend (updated): adds `/analyze-text`

Same NLP pipeline as `/analyze`, but takes typed text directly from the Journal tab's textbox instead of requiring a file upload.

In [ ]:
%%writefile backend.py
import os, io, jwt, csv, logging
from fastapi import FastAPI, UploadFile, File, Form, Header, HTTPException
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv
from nlp_pipeline import process_employee_feedback, wellness_chat_reply

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("backend")

load_dotenv()

SECRET = os.getenv("JWT_SECRET")
if not SECRET:
    logger.error("❌ JWT_SECRET not found in environment variables!")
else:
    logger.info(f"✅ JWT_SECRET loaded (length: {len(SECRET)})")

app = FastAPI(title="Upload API")

app.add_middleware(CORSMiddleware, allow_origins=["*"],
                    allow_methods=["*"], allow_headers=["*"])

def get_user(authorization: str = Header(None)):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(401, "Missing token")
    token = authorization.split(" ", 1)[1]
    try:
        return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError as e:
        logger.warning(f"Auth failed: {str(e)}")
        raise HTTPException(401, f"Invalid or expired token: {str(e)}")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/upload")
async def upload(file: UploadFile = File(...), authorization: str = Header(None)):
    user = get_user(authorization)

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024  # 5 MB cap
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    lines = text.splitlines()
    row_count = len(lines)
    preview_lines = lines[:20]

    columns = None
    preview_rows = None
    if ext == "csv":
        reader = csv.reader(io.StringIO(text))
        rows = list(reader)
        if rows:
            columns = rows[0]
            preview_rows = rows[1:21]
            row_count = max(len(rows) - 1, 0)

    return {
        "filename": name,
        "type": ext,
        "uploaded_by": user["username"],
        "row_count": row_count,
        "columns": columns,
        "preview_rows": preview_rows,
        "preview_lines": None if ext == "csv" else preview_lines,
    }

def _extract_text_blob(raw: bytes, ext: str, column: str | None) -> tuple[str, str | None]:
    text = raw.decode("utf-8")
    if ext == "txt":
        return text.strip(), None
    reader = csv.reader(io.StringIO(text))
    rows = list(reader)
    if not rows: raise HTTPException(400, "CSV file has no rows.")
    header = rows[0]
    data_rows = rows[1:]
    if not data_rows: raise HTTPException(400, "CSV file has a header but no data rows.")
    col_index = header.index(column) if column and column in header else len(header) - 1
    values = [row[col_index] for row in data_rows if len(row) > col_index and row[col_index].strip()]
    blob = " ".join(values).strip()
    if not blob: raise HTTPException(400, f"Column '{header[col_index]}' has no readable text.")
    return blob, header[col_index]

@app.post("/analyze")
async def analyze(file: UploadFile = File(...), column: str = Form(None), authorization: str = Header(None)):
    get_user(authorization)
    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"): raise HTTPException(400, "Only .csv or .txt files are allowed.")
    raw = await file.read()
    try:
        text_blob, used_column = _extract_text_blob(raw, ext, column)
    except UnicodeDecodeError: raise HTTPException(400, "File must be UTF-8 text.")
    results = process_employee_feedback(text_blob)
    results.update({"filename": name, "file_type": ext.upper(), "used_column": used_column})
    return results

class TextIn(BaseModel):
    text: str

@app.post("/analyze-text")
async def analyze_text(payload: TextIn, authorization: str = Header(None)):
    get_user(authorization)
    text_blob = payload.text.strip()
    if not text_blob: raise HTTPException(400, "Text cannot be empty.")
    results = process_employee_feedback(text_blob)
    results.update({"filename": None, "file_type": "TEXT", "used_column": None})
    return results

class ChatTurn(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    message: str
    history: list[ChatTurn] = []

@app.post("/chat")
async def chat(payload: ChatRequest, authorization: str = Header(None)):
    get_user(authorization)
    message = payload.message.strip()
    if not message: raise HTTPException(400, "Message cannot be empty.")
    history = [turn.dict() for turn in payload.history]
    return wellness_chat_reply(message, history=history)

Writing backend.py


In [ ]:
import logging
from db import init_db, cursor

# Set up logging to capture connection details
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("DB_INIT")

try:
    logger.info("Attempting to connect to the database...")
    # Test a simple query first
    with cursor() as cur:
        cur.execute("SELECT version();")
        version = cur.fetchone()
        logger.info(f"✅ Database Connection Successful! Version: {version['version']}")

    # Initialize tables
    init_db()
    logger.info("✅ Database tables verified/initialized successfully.")
    print("\n--- Database Check Summary ---")
    print("Status: Connected")
    print("Action: Tables verified")

except Exception as e:
    logger.error(f"❌ Database Connection Failed: {str(e)}")
    print(f"\nERROR: Could not connect to your database.")
    print(f"Please verify your DB_HOST, DB_USER, and DB_PASSWORD in the Colab Secrets.")


--- Database Check Summary ---
Status: Connected
Action: Tables verified


In [ ]:
from pyngrok import ngrok, conf
import subprocess, time, os, requests

conf.get_default().auth_token = values["NGROK_AUTHTOKEN"]

# Clear stale processes
print("🧹 Cleaning up stale processes...")
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
time.sleep(3)

print("🗴 Starting Backend (FastAPI)...")
get_ipython().system_raw('uvicorn backend:app --host 0.0.0.0 --port 8000 > backend.log 2>&1 &')
time.sleep(12)

print("🗴 Starting Frontend (Streamlit)...")
get_ipython().system_raw('streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false > streamlit.log 2>&1 &')
time.sleep(8)

# Verification Step
print("\n🔍 Status Check:")
try:
    be_check = requests.get("http://localhost:8000/health", timeout=5)
    print(f"✅ Backend: ONLINE (Status {be_check.status_code})")
except Exception:
    print("❌ Backend: OFFLINE (Check backend.log)")

try:
    fe_check = requests.get("http://localhost:8501", timeout=5)
    print(f"✅ Frontend: ONLINE (Status {fe_check.status_code})")
except Exception:
    print("❌ Frontend: OFFLINE (Check streamlit.log)")

try:
    public_url = ngrok.connect(8501, "http")
    print(f"\n🚀 LIVE ACCESS LINK: {public_url.public_url}")
except Exception as e:
    print(f"\n❌ Ngrok Error: {e}")

🧹 Cleaning up stale processes...
🗴 Starting Backend (FastAPI)...
🗴 Starting Frontend (Streamlit)...

🔍 Status Check:
✅ Backend: ONLINE (Status 200)
✅ Frontend: ONLINE (Status 200)

🚀 LIVE ACCESS LINK: https://kung-coastland-coeditor.ngrok-free.dev


In [ ]:
# Checking the backend logs to diagnose why it is offline
with open('backend.log', 'r') as f:
    print(f.read())

In [ ]:
from pyngrok import ngrok
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
print("Stopped Streamlit, FastAPI, and closed ngrok tunnel.")

Stopped Streamlit, FastAPI, and closed ngrok tunnel.
